## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
# Google Drive mounting is intentionally not used in this repository.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os
import matplotlib.pyplot as plt
import re
import string
import math

In [ ]:
# Load the file
df = pd.read_csv('data/processed/Final_Master_results_with_features.csv')

# Define models and their SHAP position column mappings
models = ['BART', 'T5', 'SciBERT', 'RoBERTa', 'DeepSeek', 'LLaMA3']

dist_map = {
    'BART': 'BART_shap_pos_abs_dist',
    'T5': 'T5_shap_pos_abs_dist',
    'SciBERT': 'SCIBERT_shap_pos_abs_dist',
    'RoBERTa': 'ROBERTA_shap_pos_abs_dist',
    'DeepSeek': 'DEEPSEEK_shap_pos_abs_dist',
    'LLaMA3': 'LLAMA3_shap_pos_abs_dist'
}

# Pre-processing: Create target variables (is_error)
for m in models:
    df[f'{m}_is_error'] = 1 - df[f'{m}_is_correct']

In [ ]:
# --- STEP 1: General Logistic Regression (All Samples) ---
print("--- STEP 1: General Logistic Regression Results ---")

# ------------------------------------------------------------
# Negation normalization for HI (P95-based, clipped to [0,1])
# Negation_Norm = min( Negation_Count / P95(Negation_Count), 1 )
# ------------------------------------------------------------

neg_p95 = df['Negation_Count'].quantile(0.95)

if pd.isna(neg_p95) or neg_p95 == 0:
    df['Negation_Norm'] = 0.0
else:
    df['Negation_Norm'] = (df['Negation_Count'] / neg_p95).clip(0, 1)



general_results = []
for m in models:
    formula = f"{m}_is_error ~ is_citation + Sentence_Length + Negation_Norm + Semantic_Depth"
    try:
        res = smf.logit(formula, data=df).fit(disp=0)
        summary = {
            'Model': m,
            'Intercept_B': res.params['Intercept'],
            'is_citation_B': res.params['is_citation'],
            'is_citation_p': res.pvalues['is_citation'],
            'Length_B': res.params['Sentence_Length'],
            'Length_p': res.pvalues['Sentence_Length'],
            'Negation_B': res.params['Negation_Norm'],
            'Negation_p': res.pvalues['Negation_Norm'],
            'Depth_B': res.params['Semantic_Depth'],
            'Depth_p': res.pvalues['Semantic_Depth']
        }
        general_results.append(summary)
    except Exception as e:
        print(f"General Model for {m} failed: {e}")

general_summary_df = pd.DataFrame(general_results)
print(general_summary_df)

In [ ]:
# --- STEP 2: Specific Logistic Regression (Single Citations) ---
print("\n--- STEP 2: Specific Logistic Regression (Single Citations + Gap) ---")
mask = df[(df['is_citation'] == 1) & (df['Multi_Citation_Flag'] == 0)].copy()

specific_results = []
df_fit = mask.copy() # Initialize df_fit once outside the loop


for m in models:

    gap_col = dist_map[m]

    # Logistic regression uses RAW Negation_Count (normalization here).
    formula = f"{m}_is_error ~ {gap_col} * Semantic_Depth + Negation_Norm + is_narrative + Sentence_Length"

    try:
        res = smf.logit(formula, data=df_fit).fit(disp=0)

        summary = {
            'Model': m,
            'Intercept_B': res.params['Intercept'],
            'Gap_B': res.params[gap_col],
            'Gap_p': res.pvalues[gap_col],
            'Depth_B': res.params['Semantic_Depth'],
            'Depth_p': res.pvalues['Semantic_Depth'],
            'Interaction_B': res.params[f'{gap_col}:Semantic_Depth'],
            'Interaction_p': res.pvalues[f'{gap_col}:Semantic_Depth'],
            'Negation_B': res.params['Negation_Norm'],
            'Negation_p': res.pvalues['Negation_Norm'],
            'Narrative_B': res.params['is_narrative'],
            'Narrative_p': res.pvalues['is_narrative'],
            'Length_B': res.params['Sentence_Length'],
            'Length_p': res.pvalues['Sentence_Length']
        }
        specific_results.append(summary)

        # --- STEP 3: Hallucination Index (HI) Calculation ---
        # HI_Score = B1*Gap + B2*Depth + B3*Negation_Norm
        # Note: We use the absolute coefficients (or raw, but usually raw for Logit link)
        b1 = res.params[gap_col]
        b2 = res.params['Semantic_Depth']
        b3 = res.params[f'{gap_col}:Semantic_Depth']
        b4 = res.params['Negation_Norm']

        df_fit[f'{m}_HI_Score'] = (b1 * df_fit[gap_col]) + \
                                      (b2 * df_fit['Semantic_Depth']) + \
                                      (b4 * df_fit['Negation_Norm'])  # <-- normalized for HI.

        # HI_Score = B1*Gap + B2*Depth + B3*(Gap*Depth) + B4*Negation_Norm
        """df_fit[f'{m}_HI_Score'] = (b1 * df_fit[gap_col]) + \
                                      (b2 * df_fit['Semantic_Depth']) + \
                                      (b3 * (df_fit[gap_col] * df_fit['Semantic_Depth'])) + \
                                      (b4 * df_fit['Negation_Norm'])  # <-- normalized for HI."""
    except Exception as e:
        print(f"Specific Model for {m} failed: {e}")

specific_summary_df = pd.DataFrame(specific_results)
print(specific_summary_df)

# Export final files
general_summary_df.to_csv('outputs/results/Step1_General_Coefficients.csv', index=False)
specific_summary_df.to_csv('outputs/results/Step2_Specific_Coefficients.csv', index=False)
df_fit.to_csv('data/processed/Step3_Final_Dataset_with_HI_Scores.csv', index=False)

print("\nFiles 'Step1_General_Coefficients.csv', 'Step2_Specific_Coefficients.csv', and 'Step3_Final_Dataset_with_HI_Scores.csv' saved.")

Global HPI

In [ ]:
import statsmodels.api as sm

# ------------------------------------------------------------
# 0) Load data
# ------------------------------------------------------------
path = "data/processed/Step3_Final_Dataset_with_HI_Scores.csv"
df = pd.read_csv(path)

# ------------------------------------------------------------
# 1) Columns
# ------------------------------------------------------------
dist_cols = [
    "BART_shap_pos_abs_dist",
    "T5_shap_pos_abs_dist",
    "SCIBERT_shap_pos_abs_dist",
    "ROBERTA_shap_pos_abs_dist",
    "DEEPSEEK_shap_pos_abs_dist",
    "LLAMA3_shap_pos_abs_dist",
]

error_cols = [
    "BART_is_error",
    "T5_is_error",
    "SciBERT_is_error",
    "RoBERTa_is_error",
    "DeepSeek_is_error",
    "LLaMA3_is_error",
]

required_cols = ["Semantic_Depth", "Negation_Norm"] + dist_cols + error_cols
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# ------------------------------------------------------------
# 2) Build a TRUE Global Gap (sentence-level, model-agnostic)
#    Robust choice: median across models
#    (If you prefer mean, see the commented line)
# ------------------------------------------------------------
df["Global_Gap"] = df[dist_cols].median(axis=1, skipna=True)
# df["Global_Gap"] = df[dist_cols].mean(axis=1, skipna=True)

# Basic cleaning: ensure numeric
for c in ["Global_Gap", "Semantic_Depth", "Negation_Norm"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows where core features are missing
df_base = df.dropna(subset=["Global_Gap", "Semantic_Depth", "Negation_Norm"]).copy()

# ------------------------------------------------------------
# 3) Create long-format dataset: one row per (sentence, model)
#    We'll keep the same Global_Gap/Depth/Negation for all models,
#    and bring in each model's error label.
# ------------------------------------------------------------
# Assign a stable sentence id (or use your existing id column if you have one)
df_base = df_base.reset_index(drop=True)
df_base["sentence_id"] = df_base.index

# Melt errors into long format
long_err = df_base.melt(
    id_vars=["sentence_id", "Global_Gap", "Semantic_Depth", "Negation_Norm"],
    value_vars=error_cols,
    var_name="model_error_col",
    value_name="is_error",
)

# Map model name from column
def col_to_model(colname: str) -> str:
    return colname.replace("_is_error", "")

long_err["model"] = long_err["model_error_col"].apply(col_to_model)

# Ensure binary numeric
long_err["is_error"] = pd.to_numeric(long_err["is_error"], errors="coerce")

# Drop missing error labels
long_err = long_err.dropna(subset=["is_error"]).copy()
# Force 0/1
long_err["is_error"] = (long_err["is_error"] > 0).astype(int)

# ------------------------------------------------------------
# 4) Global Logistic Regression (pooled across models)
#    Error ~ Global_Gap + Semantic_Depth + Negation_Norm
# ------------------------------------------------------------
X = long_err[["Global_Gap", "Semantic_Depth", "Negation_Norm"]].copy()
X = sm.add_constant(X)  # intercept
y = long_err["is_error"]

# Fit logistic regression
logit = sm.Logit(y, X)
res = logit.fit(disp=False)

print("\n=== GLOBAL Logistic Regression Results ===")
print(res.summary())

b0 = res.params["const"]
b1 = res.params["Global_Gap"]
b2 = res.params["Semantic_Depth"]
b3 = res.params["Negation_Norm"]

print("\nGlobal coefficients:")
print(f"Intercept (b0): {b0:.6f}")
print(f"b1 (Global_Gap): {b1:.6f}")
print(f"b2 (Semantic_Depth): {b2:.6f}")
print(f"b3 (Negation_Norm): {b3:.6f}")

# ------------------------------------------------------------
# 5) Compute Global HPI per sentence (single value per sentence)
# ------------------------------------------------------------
df_base["HPI_global"] = (
    b1 * df_base["Global_Gap"]
    + b2 * df_base["Semantic_Depth"]
    + b3 * df_base["Negation_Norm"]
)

# If you want the intercept included in the score (usually you don't need it):
# df_base["HPI_global_with_intercept"] = b0 + df_base["HPI_global"]

print("\nHPI_global stats:")
print(df_base["HPI_global"].describe())

# ------------------------------------------------------------
# 6) OPTIONAL: Robustness comparison per model on SAME HPI scale
#    - Bin HPI_global and compute error rate per model per bin
# ------------------------------------------------------------
# Merge back HPI_global into long_err
long_err = long_err.merge(
    df_base[["sentence_id", "HPI_global"]],
    on="sentence_id",
    how="left"
).dropna(subset=["HPI_global"])

# Create bins (quantiles) of HPI_global
n_bins = 5  # change to 3/5/10
long_err["HPI_bin"] = pd.qcut(long_err["HPI_global"], q=n_bins, duplicates="drop")

bin_summary = (
    long_err.groupby(["model", "HPI_bin"])["is_error"]
    .agg(error_rate="mean", n="count")
    .reset_index()
    .sort_values(["model", "HPI_bin"])
)

print("\n=== Error rate per model across GLOBAL HPI bins ===")
print(bin_summary)

# ------------------------------------------------------------
# 7) Save outputs (optional)
# ------------------------------------------------------------
out_sentence = "data/processed/dataset_with_HPI_global_sentence_level.csv"
out_long = "data/processed/longformat_with_HPI_global.csv"

df_out = df.copy()
# merge sentence-level HPI back to original rows by index mapping
# If df had been filtered, align by original row index:
# easiest: compute HPI_global on df itself for rows where features exist
df_out["Global_Gap"] = df[dist_cols].median(axis=1, skipna=True)
for c in ["Global_Gap", "Semantic_Depth", "Negation_Norm"]:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce")

mask_ok = df_out[["Global_Gap","Semantic_Depth","Negation_Norm"]].notna().all(axis=1)
df_out.loc[mask_ok, "HPI_global"] = (
    b1 * df_out.loc[mask_ok, "Global_Gap"]
    + b2 * df_out.loc[mask_ok, "Semantic_Depth"]
    + b3 * df_out.loc[mask_ok, "Negation_Norm"]
)

df_out.to_csv(out_sentence, index=False)
long_err.to_csv(out_long, index=False)

print("\nSaved:")
print(out_sentence)
print(out_long)

υπολογίζει:

Thresholds P95 (Case 4) & P75 (Case 3) ανά μοντέλο πάνω στον ίδιο Global HPI

Case 1–4 re-definition με Global HPI, με threshold T_m ανά μοντέλο (που προκύπτει από τα percentiles)

Summary counts/means ανά Case και μοντέλο

(Optional) Export σε CSV

Σημαντικό: Επειδή τώρα το HPI είναι global, το “ποια είναι η πίεση” είναι κοινό.
Το “πότε αρχίζει να σπάει” είναι model-specific → γι’ αυτό βγάζουμε thresholds ανά μοντέλο.

In [ ]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Assumes you already have:
#   - df_base with columns: sentence_id, HPI_global
#   - long_err with columns: sentence_id, model, is_error, HPI_global
# If not, load them from the saved CSVs created previously:
#long_err = pd.read_csv("data/processed/longformat_with_HPI_global.csv")
# ------------------------------------------------------------

# Safety checks
for col in ["sentence_id", "model", "is_error", "HPI_global"]:
    if col not in long_err.columns:
        raise ValueError(f"Missing '{col}' in long_err")

long_err["is_error"] = pd.to_numeric(long_err["is_error"], errors="coerce").fillna(0).astype(int)
long_err["HPI_global"] = pd.to_numeric(long_err["HPI_global"], errors="coerce")
long_err = long_err.dropna(subset=["HPI_global"]).copy()

# ------------------------------------------------------------
# 1) Percentile thresholds per model:
#    - P95 of HPI_global among CORRECT predictions (Case 4-like region)
#    - P75 of HPI_global among ERRORS (Case 3-like region)
# ------------------------------------------------------------
def safe_quantile(x, q):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return np.nan
    return x.quantile(q)

threshold_rows = []
for m, g in long_err.groupby("model"):
    hpi_correct = g.loc[g["is_error"] == 0, "HPI_global"]
    hpi_error   = g.loc[g["is_error"] == 1, "HPI_global"]

    p95_case4 = safe_quantile(hpi_correct, 0.95)  # "robust upper bound" under correctness
    p75_case3 = safe_quantile(hpi_error,   0.75)  # "failure onset" for errors

    n_correct = (g["is_error"] == 0).sum()
    n_error   = (g["is_error"] == 1).sum()

    threshold_rows.append({
        "model": m,
        "n_correct": n_correct,
        "n_error": n_error,
        "HPI_P95_correct": p95_case4,
        "HPI_P75_error": p75_case3
    })

thr_df = pd.DataFrame(threshold_rows).sort_values("model")
print("\n=== Global-HPI Percentile Thresholds per Model ===")
print(thr_df)

# ------------------------------------------------------------
# 2) Choose a model-specific threshold T_m on the GLOBAL HPI axis
#    This defines the boundary between "low pressure" and "high pressure"
#    Options:
#      A) T_m = P75_error  (onset of systematic failure)
#      B) T_m = P95_correct (upper robust bound)
#      C) T_m = average of both (balanced boundary)
# ------------------------------------------------------------
thr_df["T_p75_error"] = thr_df["HPI_P75_error"]
thr_df["T_p95_correct"] = thr_df["HPI_P95_correct"]

# Balanced threshold (recommended if both exist)
thr_df["T_balanced"] = thr_df[["T_p75_error", "T_p95_correct"]].mean(axis=1)

# Pick one:
THRESHOLD_MODE = "T_p75_error"     # or "T_p95_correct" or "T_balanced"
thr_df["T_used"] = thr_df[THRESHOLD_MODE]

# Map thresholds to long_err
thr_map = dict(zip(thr_df["model"], thr_df["T_used"]))
long_err["T_used"] = long_err["model"].map(thr_map)

# Remove models with missing thresholds
long_err_cases = long_err.dropna(subset=["T_used"]).copy()

# ------------------------------------------------------------
# 3) Re-define Case 1-4 using GLOBAL HPI and model-specific threshold T_m
#    - Low pressure region:  HPI_global <= T_m
#    - High pressure region: HPI_global >  T_m
#
#    Case 1: Low pressure & Error
#    Case 2: Low pressure & Correct
#    Case 3: High pressure & Error
#    Case 4: High pressure & Correct
# ------------------------------------------------------------
low_pressure = long_err_cases["HPI_global"] <= long_err_cases["T_used"]
is_error = long_err_cases["is_error"] == 1

long_err_cases["case_id"] = np.select(
    [
        low_pressure & is_error,
        low_pressure & (~is_error),
        (~low_pressure) & is_error,
        (~low_pressure) & (~is_error)
    ],
    [
        "Case 1 (Unexpected Error)",
        "Case 2 (Protective)",
        "Case 3 (Pressure-induced Failure)",
        "Case 4 (Robust under Pressure)"
    ],
    default="Unknown"
)

# ------------------------------------------------------------
# 4) Summaries per model & case
# ------------------------------------------------------------
case_summary = (
    long_err_cases
    .groupby(["model", "case_id"])
    .agg(
        Count=("case_id", "size"),
        Mean_HPI=("HPI_global", "mean"),
        P75_HPI=("HPI_global", lambda x: safe_quantile(x, 0.75)),
        P95_HPI=("HPI_global", lambda x: safe_quantile(x, 0.95)),
    )
    .reset_index()
    .sort_values(["model", "case_id"])
)

print("\n=== Case Summary (Global HPI + model-specific threshold) ===")
print(case_summary)

# ------------------------------------------------------------
# 5) Case Summary + Features (no merge needed)
# ------------------------------------------------------------
case_summary_feat = (
    long_err_cases
    .groupby(["model", "case_id"])
    .agg(
        Count=("case_id", "size"),
        Mean_HPI=("HPI_global", "mean"),
        Mean_Gap=("Global_Gap", "mean"),
        Mean_Depth=("Semantic_Depth", "mean"),
        Mean_Negation=("Negation_Norm", "mean"),
    )
    .reset_index()
    .sort_values(["model", "case_id"])
)

print("\n=== Case Summary + Features ===")
print(case_summary_feat)

# ------------------------------------------------------------
# 6) Save outputs (optional)
# ------------------------------------------------------------
out_thr = "outputs/results/globalHPI_thresholds_per_model.csv"
out_cases = "data/processed/longformat_with_globalHPI_cases.csv"
out_case_summary = "outputs/results/globalHPI_case_summary.csv"

thr_df.to_csv(out_thr, index=False)
long_err_cases.to_csv(out_cases, index=False)
case_summary.to_csv(out_case_summary, index=False)

print("\nSaved:")
print(out_thr)
print(out_cases)
print(out_case_summary)

Boxplots (Case 4 vs Case 3) ανά μοντέλο με κοινό y-axis (ίδιο Global HPI scale)

Curves: error rate vs Global HPI (bins) ανά μοντέλο (πολύ “robustness plot”)

In [ ]:
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# INPUT: long_err_cases (from previous step)
# Must have: model, case_id, HPI_global
# ------------------------------------------------------------
needed = {"model", "case_id", "HPI_global"}
if not needed.issubset(set(long_err_cases.columns)):
    raise ValueError(f"long_err_cases missing columns: {needed - set(long_err_cases.columns)}")

dfp = long_err_cases.copy()
dfp["HPI_global"] = pd.to_numeric(dfp["HPI_global"], errors="coerce")
dfp = dfp.dropna(subset=["HPI_global"])

# Keep only Case 3 and Case 4
case3_name = "Case 3 (Pressure-induced Failure)"
case4_name = "Case 4 (Robust under Pressure)"
dfp = dfp[dfp["case_id"].isin([case3_name, case4_name])].copy()

# Global y-limits across ALL models (shared axis)
y_min = dfp["HPI_global"].quantile(0.01)
y_max = dfp["HPI_global"].quantile(0.99)

# Optional: "nice" upper bound
step = 0.5
y_max_nice = np.ceil(y_max / step) * step

models = sorted(dfp["model"].unique())

def plot_case34_boxplots_per_model(dfp, models, show_points=True, out_dir=None):
    if out_dir is not None:
        import os
        os.makedirs(out_dir, exist_ok=True)

    for m in models:
        dm = dfp[dfp["model"] == m].copy()
        d3 = dm[dm["case_id"] == case3_name]["HPI_global"].values
        d4 = dm[dm["case_id"] == case4_name]["HPI_global"].values

        plt.figure(figsize=(6.5, 4.2))
        data = [d4, d3]  # order: Case 4, Case 3

        # Boxplot
        plt.boxplot(
            data,
            tick_labels=["Case 4 (Correct)", "Case 3 (Error)"],
            showfliers=True
        )

        # Optional: overlay jitter points
        if show_points:
            rng = np.random.default_rng(42)
            for i, vals in enumerate(data, start=1):
                if len(vals) == 0:
                    continue
                x = i + rng.uniform(-0.08, 0.08, size=len(vals))
                plt.scatter(x, vals, s=10)

        plt.title(f"{m}: Global HPI distribution (Case 4 vs Case 3)")
        plt.ylabel("HPI_global")
        plt.ylim(y_min, y_max_nice)
        plt.grid(True, axis="y", alpha=0.3)

        # Quick stats on plot (optional)
        n4, n3 = len(d4), len(d3)
        plt.text(1.02, y_max_nice*0.98, f"n4={n4}\nn3={n3}", va="top")

        plt.tight_layout()

        if out_dir is not None:
            plt.savefig(f"{out_dir}/{m}_GlobalHPI_Case4_vs_Case3_boxplot.png", dpi=200)
            plt.savefig(f"{out_dir}/{m}_GlobalHPI_Case4_vs_Case3_boxplot.pdf")
        plt.show()

# Run
plot_case34_boxplots_per_model(dfp, models, show_points=True, out_dir="outputs/figures/global_hpi_case3_4_plots")

box plot 2x3

In [ ]:
# ------------------------------------------------------------
# INPUT: long_err_cases must include: model, case_id, HPI_global
# ------------------------------------------------------------
need = {"model", "case_id", "HPI_global"}
if not need.issubset(set(long_err_cases.columns)):
    raise ValueError(f"long_err_cases missing columns: {need - set(long_err_cases.columns)}")

dfp = long_err_cases.copy()
dfp["HPI_global"] = pd.to_numeric(dfp["HPI_global"], errors="coerce")
dfp = dfp.dropna(subset=["HPI_global"]).copy()

case3_name = "Case 3 (Pressure-induced Failure)"
case4_name = "Case 4 (Robust under Pressure)"
dfp = dfp[dfp["case_id"].isin([case3_name, case4_name])].copy()

# Order models (edit if you want a specific order)
default_order = ["BART", "T5", "SciBERT", "RoBERTa", "DeepSeek", "LLaMA3"]
models_present = sorted(dfp["model"].unique().tolist())

# Keep only those present, in a nice order
models = [m for m in default_order if m in models_present] + [m for m in models_present if m not in default_order]
models = models[:6]  # 2x3 = 6 models

if len(models) < 6:
    print(f"Warning: found only {len(models)} models. Panel will show {len(models)} subplots.")

# ------------------------------------------------------------
# GLOBAL y-axis limits (shared across all subplots)
# ------------------------------------------------------------
y_min = dfp["HPI_global"].quantile(0.01)
y_max = dfp["HPI_global"].quantile(0.99)

# Make top limit "nice"
step = 0.5
y_max = np.ceil(y_max / step) * step

# ------------------------------------------------------------
# Panel plot: 2x3
# ------------------------------------------------------------
SHOW_POINTS = True        # jitter points overlay
POINT_SIZE = 7
RANDOM_SEED = 42
SAVE_PATH_PNG = "outputs/figures/global_hpi_case3_4_plots/GlobalHPI_Case4_vs_Case3_2x3_panel.png"
SAVE_PATH_PDF = "outputs/figures/global_hpi_case3_4_plots/GlobalHPI_Case4_vs_Case3_2x3_panel.pdf"

rng = np.random.default_rng(RANDOM_SEED)

fig, axes = plt.subplots(2, 3, figsize=(9, 10), sharey=True)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(models):
        ax.axis("off")
        continue

    m = models[i]
    dm = dfp[dfp["model"] == m]

    d4 = dm.loc[dm["case_id"] == case4_name, "HPI_global"].values
    d3 = dm.loc[dm["case_id"] == case3_name, "HPI_global"].values

    data = [d4, d3]  # Case 4, Case 3

    ax.boxplot(
        data,
        tick_labels=["C4", "C3"],
        showfliers=True
    )

    if SHOW_POINTS:
        for xpos, vals in enumerate(data, start=1):
            if len(vals) == 0:
                continue
            x = xpos + rng.uniform(-0.10, 0.10, size=len(vals))
            ax.scatter(x, vals, s=POINT_SIZE)

    # ---- Percentiles ----
    if len(d4) > 0:
        p95_c4 = np.percentile(d4, 95)
        ax.axhline(p95_c4, linestyle="--")
        ax.text(1.1, p95_c4, "P95 C4", va="bottom", fontsize=10)

    if len(d3) > 0:
        p75_c3 = np.percentile(d3, 75)
        ax.axhline(p75_c3, linestyle=":")
        ax.text(1.6, p75_c3, "P75 C3", va="bottom", fontsize=10)

    ax.set_title(m)
    ax.set_ylim(y_min, y_max)
    ax.grid(True, axis="y", alpha=0.25)

    # annotate sample sizes
    ax.text(
        0.5, 0.01, # 0.98, 0.98,
        f"C4: n={len(d4)}\nC3: n={len(d3)}",
        transform=ax.transAxes,
        ha="center", va="bottom"  # ha="right", va="top"
    )

# Axis labels (single for figure)
fig.suptitle("Global HPI - Model Specific Threshold", y=1.02, fontsize=15)
fig.text(0.03, 0.49, "Global HPI", va="center", rotation="vertical", fontsize=10)
fig.text(0.5, 0.02, "",
         ha="center")

plt.tight_layout()

# Save
fig.savefig(SAVE_PATH_PNG, dpi=220, bbox_inches="tight")
fig.savefig(SAVE_PATH_PDF, bbox_inches="tight")

plt.show()

print("Saved:")
print(SAVE_PATH_PNG)
print(SAVE_PATH_PDF)

Main comparison with global common threshold.
Αυτό να είναι το βασικό συγκριτικό αποτέλεσμα:

ποια μοντέλα κρατούν περισσότερα σωστά στο C4
Με ενιαίο threshold:

η σύγκριση γίνεται πιο καθαρή και πιο υπερασπίσιμη στο paper

ειδικά αν θελουμε να πουμε ότι ο Global HPI ορίζει κοινή πίεση για όλα τα μοντέλα

In [ ]:
# ------------------------------------------------------------
# Assumes you already have:
#   long_err with columns:
#   sentence_id, model, is_error, HPI_global
# ------------------------------------------------------------

# Safety checks
for col in ["sentence_id", "model", "is_error", "HPI_global"]:
    if col not in long_err.columns:
        raise ValueError(f"Missing '{col}' in long_err")

long_err = long_err.copy()
long_err["is_error"] = pd.to_numeric(long_err["is_error"], errors="coerce").fillna(0).astype(int)
long_err["HPI_global"] = pd.to_numeric(long_err["HPI_global"], errors="coerce")
long_err = long_err.dropna(subset=["HPI_global"]).copy()

# ------------------------------------------------------------
# 1) ONE COMMON GLOBAL THRESHOLD FOR ALL MODELS
# ------------------------------------------------------------
# Choose how you want to define the single threshold:
# Option A: 75th percentile of ALL Global HPI values
# Option B: 80th percentile
# Option C: fixed value manually
#
# Recommended starting point:
GLOBAL_THRESHOLD_MODE = "P75_ALL"   # options: "P75_ALL", "P80_ALL", "MANUAL"

if GLOBAL_THRESHOLD_MODE == "P75_ALL":
    GLOBAL_THRESHOLD = long_err["HPI_global"].quantile(0.75)
elif GLOBAL_THRESHOLD_MODE == "P80_ALL":
    GLOBAL_THRESHOLD = long_err["HPI_global"].quantile(0.80)
elif GLOBAL_THRESHOLD_MODE == "MANUAL":
    GLOBAL_THRESHOLD = 2.50   # <- change if needed
else:
    raise ValueError("Invalid GLOBAL_THRESHOLD_MODE")

print(f"\n=== COMMON GLOBAL THRESHOLD USED FOR ALL MODELS ===")
print(f"Mode: {GLOBAL_THRESHOLD_MODE}")
print(f"Global threshold T = {GLOBAL_THRESHOLD:.4f}")

# Assign same threshold to all rows
long_err_cases = long_err.copy()
long_err_cases["T_used"] = GLOBAL_THRESHOLD

# ------------------------------------------------------------
# 2) Re-define Case 1-4 using the SAME threshold for all models
#
#    Low pressure region:  HPI_global <= T
#    High pressure region: HPI_global >  T
#
#    Case 1: Low pressure & Error
#    Case 2: Low pressure & Correct
#    Case 3: High pressure & Error
#    Case 4: High pressure & Correct
# ------------------------------------------------------------
low_pressure = long_err_cases["HPI_global"] <= long_err_cases["T_used"]
is_error = long_err_cases["is_error"] == 1

long_err_cases["case_id"] = np.select(
    [
        low_pressure & is_error,
        low_pressure & (~is_error),
        (~low_pressure) & is_error,
        (~low_pressure) & (~is_error)
    ],
    [
        "Case 1 (Unexpected Error)",
        "Case 2 (Protective)",
        "Case 3 (Pressure-induced Failure)",
        "Case 4 (Robust under Pressure)"
    ],
    default="Unknown"
)

# ------------------------------------------------------------
# 3) Summary per model & case
# ------------------------------------------------------------
def safe_quantile(x, q):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return np.nan
    return x.quantile(q)

case_summary = (
    long_err_cases
    .groupby(["model", "case_id"])
    .agg(
        Count=("case_id", "size"),
        Mean_HPI=("HPI_global", "mean"),
        Median_HPI=("HPI_global", "median"),
        P75_HPI=("HPI_global", lambda x: safe_quantile(x, 0.75)),
        P95_HPI=("HPI_global", lambda x: safe_quantile(x, 0.95)),
        Min_HPI=("HPI_global", "min"),
        Max_HPI=("HPI_global", "max"),
    )
    .reset_index()
    .sort_values(["model", "case_id"])
)

print("\n=== Case Summary (COMMON Global Threshold) ===")
print(case_summary)

# ------------------------------------------------------------
# 4) Feature summary (optional)
# ------------------------------------------------------------
feature_cols = ["Global_Gap", "Semantic_Depth", "Negation_Norm"]
existing_feature_cols = [c for c in feature_cols if c in long_err_cases.columns]

agg_dict = {
    "Count": ("case_id", "size"),
    "Mean_HPI": ("HPI_global", "mean")
}
for c in existing_feature_cols:
    agg_dict[f"Mean_{c}"] = (c, "mean")

case_summary_feat = (
    long_err_cases
    .groupby(["model", "case_id"])
    .agg(**agg_dict)
    .reset_index()
    .sort_values(["model", "case_id"])
)

print("\n=== Case Summary + Features (COMMON Global Threshold) ===")
print(case_summary_feat)

# ------------------------------------------------------------
# 5) Save outputs
# ------------------------------------------------------------
out_cases = "data/processed/longformat_with_globalHPI_cases_COMMON_THRESHOLD.csv"
out_case_summary = "outputs/results/globalHPI_case_summary_COMMON_THRESHOLD.csv"

long_err_cases.to_csv(out_cases, index=False)
case_summary.to_csv(out_case_summary, index=False)

print("\nSaved:")
print(out_cases)
print(out_case_summary)

Plot για C4 vs C3 με ενιαίο threshold. Πλέον τα C3/C4 έχουν βγει με το ίδιο threshold για όλους.

In [ ]:
# ------------------------------------------------------------
# INPUT: long_err_cases must include: model, case_id, HPI_global, T_used
# ------------------------------------------------------------
need = {"model", "case_id", "HPI_global", "T_used"}
if not need.issubset(set(long_err_cases.columns)):
    raise ValueError(f"long_err_cases missing columns: {need - set(long_err_cases.columns)}")

dfp = long_err_cases.copy()
dfp["HPI_global"] = pd.to_numeric(dfp["HPI_global"], errors="coerce")
dfp = dfp.dropna(subset=["HPI_global"]).copy()

case3_name = "Case 3 (Pressure-induced Failure)"
case4_name = "Case 4 (Robust under Pressure)"
dfp = dfp[dfp["case_id"].isin([case3_name, case4_name])].copy()

# Common threshold (same for all models)
common_threshold = dfp["T_used"].dropna().iloc[0]

# Order models
default_order = ["BART", "T5", "SciBERT", "RoBERTa", "DeepSeek", "LLaMA3"]
models_present = sorted(dfp["model"].unique().tolist())
models = [m for m in default_order if m in models_present] + [m for m in models_present if m not in default_order]
models = models[:6]

if len(models) < 6:
    print(f"Warning: found only {len(models)} models. Panel will show {len(models)} subplots.")

# ------------------------------------------------------------
# GLOBAL y-axis limits
# ------------------------------------------------------------
y_min = dfp["HPI_global"].quantile(0.01)
y_max = dfp["HPI_global"].quantile(0.99)

step = 0.5
y_max = np.ceil(y_max / step) * step

# Optional: make lower limit start at threshold or a little below
y_min = min(y_min, common_threshold - 0.1)

# ------------------------------------------------------------
# Panel plot: 2x3
# ------------------------------------------------------------
SHOW_POINTS = True
POINT_SIZE = 7
RANDOM_SEED = 42

SAVE_PATH_PNG = "outputs/figures/global_hpi_case3_4_plots/GlobalHPI_Case4_vs_Case3_2x3_panel_COMMON_THRESHOLD.png"
SAVE_PATH_PDF = "outputs/figures/global_hpi_case3_4_plots/GlobalHPI_Case4_vs_Case3_2x3_panel_COMMON_THRESHOLD.pdf"

rng = np.random.default_rng(RANDOM_SEED)

fig, axes = plt.subplots(2, 3, figsize=(9, 10), sharey=True)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(models):
        ax.axis("off")
        continue

    m = models[i]
    dm = dfp[dfp["model"] == m]

    d4 = dm.loc[dm["case_id"] == case4_name, "HPI_global"].values
    d3 = dm.loc[dm["case_id"] == case3_name, "HPI_global"].values

    data = [d4, d3]

    ax.boxplot(
        data,
        tick_labels=["C4", "C3"],
        showfliers=True
    )

    if SHOW_POINTS:
        for xpos, vals in enumerate(data, start=1):
            if len(vals) == 0:
                continue
            x = xpos + rng.uniform(-0.10, 0.10, size=len(vals))
            ax.scatter(x, vals, s=POINT_SIZE)

    # Same threshold line for ALL models
    ax.axhline(common_threshold, linestyle="--", linewidth=1.5)
    ax.text(1.2, common_threshold, "Common T", va="bottom", fontsize=10)

    # Optional percentiles within each case
    if len(d4) > 0:
        p95_c4 = np.percentile(d4, 95)
        ax.axhline(p95_c4, linestyle="--", alpha=0.7)
        ax.text(1.15, p95_c4, "P95 C4", va="bottom", fontsize=10)

    if len(d3) > 0:
        p75_c3 = np.percentile(d3, 75)
        ax.axhline(p75_c3, linestyle=":", alpha=0.8)
        ax.text(1.55, p75_c3, "P75 C3", va="bottom", fontsize=10)

    ax.set_title(m)
    ax.set_ylim(y_min, y_max)
    ax.grid(True, axis="y", alpha=0.25)

    ax.text(
        0.5, 0.1,
        f"C4: n={len(d4)}\nC3: n={len(d3)}",
        transform=ax.transAxes,
        ha="center", va="bottom"
    )

fig.suptitle("Global HPI - Common Threshold", y=1.02, fontsize=15)
fig.text(0.03, 0.49, "Global HPI", va="center", rotation="vertical", fontsize=10)

plt.tight_layout()

fig.savefig(SAVE_PATH_PNG, dpi=220, bbox_inches="tight")
fig.savefig(SAVE_PATH_PDF, bbox_inches="tight")

plt.show()

print("Saved:")
print(SAVE_PATH_PNG)
print(SAVE_PATH_PDF)

Διερεύνηση SHAP της περίπτωσης Global HPI common threshold για C3 + c4 για τα μοντέλα.

In [ ]:
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level.csv"
OUTPUT_DIR = "outputs/results/global_common_threshold_shap_analysis"
THRESHOLD = 2.1036

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    {"model": "BART", "error_col": "BART_is_error", "token_col": "BART_shap_keyword_token"},
    {"model": "T5", "error_col": "T5_is_error", "token_col": "T5_shap_keyword_token"},
    {"model": "SciBERT", "error_col": "SciBERT_is_error", "token_col": "SCIBERT_shap_keyword_token"},
    {"model": "RoBERTa", "error_col": "RoBERTa_is_error", "token_col": "ROBERTA_shap_keyword_token"},
    {"model": "DeepSeek", "error_col": "DeepSeek_is_error", "token_col": "DEEPSEEK_shap_keyword_token"},
    {"model": "LLaMA3", "error_col": "LLaMA3_is_error", "token_col": "LLAMA3_shap_keyword_token"},
]

STOPWORDS = set("""
a an the and or but if while of at by for with about against between into through during before after above below
to from up down in out on off over under again further then once here there all any both each few more most other
some such no nor not only own same so than too very can will just don should now is are was were be been being
as this that these those it its we our they their he she you your i me my do does did have has had
""".split())

EVALUATIVE = set("""
good bad better best worse worst important interesting significant significantly improved improvement strong weak
effective ineffective robust robustly novel promising limited limitation limitations useful superior inferior successful
failure accurate inaccurate efficient inefficient positive negative neutral satisfying popular exceptional rapid
unnecessary competitive reasonable attractive accurate
""".split())

def normalize_token(token):
    if pd.isna(token):
        return None
    token = str(token).strip()
    if token == "" or token.lower() == "nan":
        return None
    return token

def token_category(token: str) -> str:
    low = token.lower()

    if all(ch in string.punctuation for ch in token):
        return "punctuation"
    if re.fullmatch(r"[\(\)\[\]\{\},.;:]+", token):
        return "punctuation"

    if low in {"no", "not", "never", "without", "n't"}:
        return "negation"

    if re.fullmatch(r"\d{1,4}[a-z]?", low) or re.search(r"\d", low):
        return "number"

    if low in STOPWORDS:
        return "stopword"

    if low in EVALUATIVE:
        return "evaluative"

    return "content/domain"

df = pd.read_csv(INPUT_CSV)

overview_rows = []
token_freq_rows = []
token_cat_rows = []

for item in MODELS:
    model = item["model"]
    error_col = item["error_col"]
    token_col = item["token_col"]

    for case_name, error_value in [("C3", 1), ("C4", 0)]:
        filtered = df[(df[error_col] == error_value) & (df["HPI_global"] > THRESHOLD)].copy()
        slim = filtered[[token_col, "Semantic_Depth"]].copy()
        slim = slim.rename(columns={token_col: "shap_token"})
        slim["shap_token"] = slim["shap_token"].map(normalize_token)
        slim = slim[slim["shap_token"].notna()].copy()

        # Save raw per-model per-case CSV
        raw_path = os.path.join(OUTPUT_DIR, f"{model}_{case_name}_tokens.csv")
        slim.to_csv(raw_path, index=False)

        # Overview
        overview_rows.append({
            "model": model,
            "case": case_name,
            "n_rows": len(slim),
            "mean_semantic_depth": round(float(slim["Semantic_Depth"].mean()), 6) if len(slim) else None
        })

        # Token frequencies
        vc = slim["shap_token"].value_counts(dropna=False)
        total = vc.sum()
        for token, count in vc.items():
            token_freq_rows.append({
                "model": model,
                "case": case_name,
                "token": token,
                "count": int(count),
                "proportion": float(count / total) if total else 0.0
            })

        # Token categories
        slim["token_category"] = slim["shap_token"].map(token_category)
        cat_vc = slim["token_category"].value_counts(dropna=False)
        cat_total = cat_vc.sum()
        for cat, count in cat_vc.items():
            token_cat_rows.append({
                "model": model,
                "case": case_name,
                "token_category": cat,
                "count": int(count),
                "proportion": float(count / cat_total) if cat_total else 0.0
            })

overview_df = pd.DataFrame(overview_rows).sort_values(["model", "case"])
token_freq_df = pd.DataFrame(token_freq_rows).sort_values(
    ["model", "case", "count", "token"],
    ascending=[True, True, False, True]
)
token_cat_df = pd.DataFrame(token_cat_rows).sort_values(
    ["model", "case", "count", "token_category"],
    ascending=[True, True, False, True]
)

overview_df.to_csv(os.path.join(OUTPUT_DIR, "overview_counts.csv"), index=False)
token_freq_df.to_csv(os.path.join(OUTPUT_DIR, "token_frequency_summary_all_models.csv"), index=False)
token_cat_df.to_csv(os.path.join(OUTPUT_DIR, "token_category_summary_all_models.csv"), index=False)

print("Done.")
print(f"Files written to: {OUTPUT_DIR}")

Στατιστικός έλεγχος για τις κατηγορίες SHAP tokens ανά μοντέλο και ανά C3/C4 στο Global HPI common threshold.

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact

INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level.csv"
OUTPUT_DIR = "outputs/results/global_common_threshold_shap_stats"
THRESHOLD = 2.1036

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    {"model": "BART", "error_col": "BART_is_error", "token_col": "BART_shap_keyword_token"},
    {"model": "T5", "error_col": "T5_is_error", "token_col": "T5_shap_keyword_token"},
    {"model": "SciBERT", "error_col": "SciBERT_is_error", "token_col": "SCIBERT_shap_keyword_token"},
    {"model": "RoBERTa", "error_col": "RoBERTa_is_error", "token_col": "ROBERTA_shap_keyword_token"},
    {"model": "DeepSeek", "error_col": "DeepSeek_is_error", "token_col": "DEEPSEEK_shap_keyword_token"},
    {"model": "LLaMA3", "error_col": "LLaMA3_is_error", "token_col": "LLAMA3_shap_keyword_token"},
]

STOPWORDS = set("""
a an the and or but if while of at by for with about against between into through during before after above below
to from up down in out on off over under again further then once here there all any both each few more most other
some such no nor not only own same so than too very can will just don should now is are was were be been being
as this that these those it its we our they their he she you your i me my do does did have has had
""".split())

EVALUATIVE = set("""
good bad better best worse worst important interesting significant significantly improved improvement strong weak
effective ineffective robust robustly novel promising limited limitation limitations useful superior inferior successful
failure accurate inaccurate efficient inefficient positive negative neutral satisfying popular exceptional rapid
unnecessary competitive reasonable attractive accurate
""".split())

def normalize_token(token):
    if pd.isna(token):
        return None
    token = str(token).strip()
    if token == "" or token.lower() == "nan":
        return None
    return token

def token_category(token: str) -> str:
    low = token.lower()
    if all(ch in string.punctuation for ch in token):
        return "punctuation"
    if re.fullmatch(r"[\(\)\[\]\{\},.;:]+", token):
        return "punctuation"
    if low in {"no", "not", "never", "without", "n't"}:
        return "negation"
    if re.fullmatch(r"\d{1,4}[a-z]?", low) or re.search(r"\d", low):
        return "number"
    if low in STOPWORDS:
        return "stopword"
    if low in EVALUATIVE:
        return "evaluative"
    return "content/domain"

def cramers_v_from_table(table: pd.DataFrame) -> float:
    chi2, _, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    if n == 0:
        return np.nan
    r, k = table.shape
    denom = n * max(min(r - 1, k - 1), 1)
    return math.sqrt(chi2 / denom)

def phi_from_2x2(table_2x2: pd.DataFrame) -> float:
    arr = table_2x2.to_numpy()
    if arr.shape != (2, 2):
        return np.nan
    a, b = arr[0, 0], arr[0, 1]
    c, d = arr[1, 0], arr[1, 1]
    denom = math.sqrt((a+b)*(c+d)*(a+c)*(b+d))
    if denom == 0:
        return np.nan
    return (a*d - b*c) / denom

def benjamini_hochberg(p_values):
    p_values = np.array(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adj = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        adj[i] = val
        prev = val
    out = np.empty(n, dtype=float)
    out[order] = adj
    return out

df = pd.read_csv(INPUT_CSV)

long_rows = []
for item in MODELS:
    model = item["model"]
    error_col = item["error_col"]
    token_col = item["token_col"]

    sub = df[df["HPI_global"] > THRESHOLD][[error_col, token_col]].copy()
    sub["case"] = np.where(sub[error_col] == 1, "C3", "C4")
    sub["token"] = sub[token_col].map(normalize_token)
    sub = sub[sub["token"].notna()].copy()
    sub["token_category"] = sub["token"].map(token_category)
    sub["model"] = model
    long_rows.append(sub[["model", "case", "token", "token_category"]])

long_df = pd.concat(long_rows, ignore_index=True)
long_df.to_csv(os.path.join(OUTPUT_DIR, "long_shap_token_categories.csv"), index=False)

per_model_rows = []
for model in long_df["model"].unique():
    t = pd.crosstab(
        long_df.loc[long_df["model"] == model, "case"],
        long_df.loc[long_df["model"] == model, "token_category"]
    )
    t = t.reindex(index=["C3", "C4"], fill_value=0)
    chi2, p, dof, expected = chi2_contingency(t)
    cramer_v = cramers_v_from_table(t)
    min_expected = expected.min() if expected.size else np.nan
    per_model_rows.append({
        "model": model,
        "test": "chi_square_independence",
        "chi2": chi2,
        "p_value": p,
        "dof": dof,
        "cramers_v": cramer_v,
        "n_total": int(t.to_numpy().sum()),
        "min_expected_count": float(min_expected),
    })
    t.reset_index().to_csv(os.path.join(OUTPUT_DIR, f"{model}_C3_vs_C4_contingency.csv"), index=False)

per_model_df = pd.DataFrame(per_model_rows).sort_values("p_value")
if len(per_model_df):
    per_model_df["p_value_bh"] = benjamini_hochberg(per_model_df["p_value"].tolist())
per_model_df.to_csv(os.path.join(OUTPUT_DIR, "per_model_C3_vs_C4_chi_square_results.csv"), index=False)

per_case_rows = []
for case in ["C3", "C4"]:
    t = pd.crosstab(
        long_df.loc[long_df["case"] == case, "model"],
        long_df.loc[long_df["case"] == case, "token_category"]
    )
    chi2, p, dof, expected = chi2_contingency(t)
    cramer_v = cramers_v_from_table(t)
    min_expected = expected.min() if expected.size else np.nan
    per_case_rows.append({
        "case": case,
        "test": "chi_square_independence",
        "chi2": chi2,
        "p_value": p,
        "dof": dof,
        "cramers_v": cramer_v,
        "n_total": int(t.to_numpy().sum()),
        "min_expected_count": float(min_expected),
    })
    t.reset_index().to_csv(os.path.join(OUTPUT_DIR, f"{case}_models_vs_categories_contingency.csv"), index=False)

per_case_df = pd.DataFrame(per_case_rows).sort_values("p_value")
if len(per_case_df):
    per_case_df["p_value_bh"] = benjamini_hochberg(per_case_df["p_value"].tolist())
per_case_df.to_csv(os.path.join(OUTPUT_DIR, "per_case_models_vs_categories_chi_square_results.csv"), index=False)

focused_rows = []
for model in long_df["model"].unique():
    sub = long_df[long_df["model"] == model].copy()
    categories = sorted(sub["token_category"].unique())
    for cat in categories:
        temp = sub.assign(in_cat=(sub["token_category"] == cat).astype(int))
        cont = pd.crosstab(temp["case"], temp["in_cat"]).reindex(index=["C3","C4"], columns=[0,1], fill_value=0)
        min_cell = cont.to_numpy().min()
        if min_cell < 5:
            oddsratio, p = fisher_exact(cont)
            test_name = "fisher_exact"
        else:
            chi2, p, _, _ = chi2_contingency(cont)
            oddsratio = np.nan
            test_name = "chi_square_2x2"
        phi = phi_from_2x2(cont)
        c3_prop = cont.loc["C3", 1] / cont.loc["C3"].sum() if cont.loc["C3"].sum() else np.nan
        c4_prop = cont.loc["C4", 1] / cont.loc["C4"].sum() if cont.loc["C4"].sum() else np.nan
        focused_rows.append({
            "model": model,
            "token_category": cat,
            "test": test_name,
            "p_value": p,
            "phi": phi,
            "odds_ratio_if_fisher": oddsratio,
            "c3_category_count": int(cont.loc["C3", 1]),
            "c4_category_count": int(cont.loc["C4", 1]),
            "c3_category_proportion": c3_prop,
            "c4_category_proportion": c4_prop,
            "direction": "higher_in_C4" if c4_prop > c3_prop else "higher_in_C3" if c3_prop > c4_prop else "equal"
        })

focused_df = pd.DataFrame(focused_rows).sort_values(["model","p_value"])
if len(focused_df):
    focused_df["p_value_bh_within_model"] = np.nan
    for model in focused_df["model"].unique():
        idx = focused_df["model"] == model
        focused_df.loc[idx, "p_value_bh_within_model"] = benjamini_hochberg(focused_df.loc[idx, "p_value"].tolist())
focused_df.to_csv(os.path.join(OUTPUT_DIR, "focused_C3_vs_C4_token_category_tests_per_model.csv"), index=False)

desc = (
    long_df.groupby(["model", "case", "token_category"])
    .size()
    .reset_index(name="count")
)
desc["total_in_model_case"] = desc.groupby(["model","case"])["count"].transform("sum")
desc["proportion"] = desc["count"] / desc["total_in_model_case"]
desc.to_csv(os.path.join(OUTPUT_DIR, "descriptive_category_proportions_by_model_case.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "README_stats.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Files produced:\n"
        "- long_shap_token_categories.csv: Long-format token-category dataset in high-pressure region.\n"
        "- per_model_C3_vs_C4_chi_square_results.csv: For each model, chi-square test comparing token-category distribution between C3 and C4.\n"
        "- per_case_models_vs_categories_chi_square_results.csv: For each case (C3/C4), chi-square test comparing token-category distribution across models.\n"
        "- focused_C3_vs_C4_token_category_tests_per_model.csv: 2x2 tests per model and token category (C3 vs C4).\n"
        "- descriptive_category_proportions_by_model_case.csv: Descriptive proportions for reporting.\n"
    )

print("Done.")
print("Output:", OUTPUT_DIR)

φορτώνει το CSV
κρατά μόνο τις positive περιπτώσεις (ground_truth == 2)
χωρίζει σε:
low-pressure: HPI_global <= 2.1036
high-pressure: HPI_global > 2.1036
για κάθε μοντέλο φτιάχνει 2x2 πίνακα
εφαρμόζει:
Chi-square test
Fisher’s exact test
υπολογίζει:
odds ratio
positive recall σε low και high
κάνει και Benjamini–Hochberg correction

Για κάθε μοντέλο:

low_positive_recall: ποσοστό σωστών positive προβλέψεων στη low-pressure περιοχή
high_positive_recall: ποσοστό σωστών positive προβλέψεων στη high-pressure περιοχή
recall_difference_high_minus_low:
αρνητικό → πέφτει η απόδοση στα positive στη high-pressure
θετικό → βελτιώνεται στη high-pressure
chi2_p / fisher_p:
αν είναι μικρότερα από 0.05, η διαφορά δεν φαίνεται τυχαία
chi2_p_bh / fisher_p_bh:
τα ίδια αλλά με Benjamini–Hochberg correction
odds_ratio_correct_low_vs_high:
> 1 → καλύτερες odds σωστής positive πρόβλεψης στη low-pressure
< 1 → καλύτερες odds σωστής positive πρόβλεψης στη high-pressure

Η στήλη recommended_test λέει:

Chi-square αν τα expected counts είναι οκ
Fisher αν κάποιο expected count είναι μικρότερο από 5

χρησιμοποιήθηκε Pearson’s chi-square
και όπου υπήρχαν μικρές αναμενόμενες συχνότητες, χρησιμοποιήθηκε Fisher’s exact test

Αυτός ο έλεγχος απαντά στο ερώτημα:
Για τις αληθινά positive περιπτώσεις, αλλάζει σημαντικά η πιθανότητα σωστής αναγνώρισης όταν περνάμε από low-pressure σε high-pressure περιοχή;

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/positive_low_vs_high_hpi_stats.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_POSITIVE = 2

# =========================
# KEEP ONLY TRUE POSITIVE-CLASS INSTANCES
# i.e. rows where the true label is positive
# =========================
pos_df = df[df["ground_truth"] == GROUND_TRUTH_POSITIVE].copy()

# Pressure groups
low_df = pos_df[pos_df["HPI_global"] <= THRESHOLD].copy()
high_df = pos_df[pos_df["HPI_global"] > THRESHOLD].copy()

print(f"Total positive instances: {len(pos_df)}")
print(f"Low-pressure positive instances: {len(low_df)}")
print(f"High-pressure positive instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct positive = model predicted 2 when ground truth is 2
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_POSITIVE).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_POSITIVE).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_POSITIVE).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_POSITIVE).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of positive class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)

    # Decide whether small expected counts exist
    min_expected = expected.min()

    # Fisher exact test for 2x2
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    # Relative recall drop/gain from low to high
    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_positive": low_correct,
        "low_wrong_positive": low_wrong,
        "high_correct_positive": high_correct,
        "high_wrong_positive": high_wrong,

        "low_total_positive": low_total,
        "high_total_positive": high_total,

        "low_positive_recall": low_recall,
        "high_positive_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct positive prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_positive_recall", "high_positive_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

print(f"\nSaved to: {OUTPUT_CSV}")

summary_cols = [
    "model",
    "low_correct_positive", "low_wrong_positive",
    "high_correct_positive", "high_wrong_positive",
    "low_positive_recall", "high_positive_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

το ίδιο για negative δίνει αυτό το αποτέλεσμα, με:

negative ground truth = 0
low-pressure: HPI_global <= 2.1036
high-pressure: HPI_global > 2.1036

Στο dataset μου υπάρχουν:

614 negative συνολικά
464 negative στη low-pressure περιοχή
150 negative στη high-pressure περιοχή

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/negative_low_vs_high_hpi_stats.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_NEGATIVE = 0

# =========================
# KEEP ONLY TRUE NEGATIVE-CLASS INSTANCES
# i.e. rows where the true label is negative
# =========================
neg_df = df[df["ground_truth"] == GROUND_TRUTH_NEGATIVE].copy()

# Pressure groups
low_df = neg_df[neg_df["HPI_global"] <= THRESHOLD].copy()
high_df = neg_df[neg_df["HPI_global"] > THRESHOLD].copy()

print(f"Total negative instances: {len(neg_df)}")
print(f"Low-pressure negative instances: {len(low_df)}")
print(f"High-pressure negative instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct negative = model predicted 0 when ground truth is 0
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_NEGATIVE).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_NEGATIVE).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_NEGATIVE).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_NEGATIVE).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of negative class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)
    min_expected = expected.min()

    # Fisher exact
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_negative": low_correct,
        "low_wrong_negative": low_wrong,
        "high_correct_negative": high_correct,
        "high_wrong_negative": high_wrong,

        "low_total_negative": low_total,
        "high_total_negative": high_total,

        "low_negative_recall": low_recall,
        "high_negative_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct negative prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_negative_recall", "high_negative_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

summary_cols = [
    "model",
    "low_correct_negative", "low_wrong_negative",
    "high_correct_negative", "high_wrong_negative",
    "low_negative_recall", "high_negative_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

print(f"\nSaved to: {OUTPUT_CSV}")

Για το neutral ισχύει:

ground truth neutral = 1
low-pressure: HPI_global <= 2.1036
high-pressure: HPI_global > 2.1036

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/neutral_low_vs_high_hpi_stats.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_NEUTRAL = 1

# =========================
# KEEP ONLY TRUE NEUTRAL-CLASS INSTANCES
# i.e. rows where the true label is neutral
# =========================
neu_df = df[df["ground_truth"] == GROUND_TRUTH_NEUTRAL].copy()

# Pressure groups
low_df = neu_df[neu_df["HPI_global"] <= THRESHOLD].copy()
high_df = neu_df[neu_df["HPI_global"] > THRESHOLD].copy()

print(f"Total neutral instances: {len(neu_df)}")
print(f"Low-pressure neutral instances: {len(low_df)}")
print(f"High-pressure neutral instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct neutral = model predicted 1 when ground truth is 1
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_NEUTRAL).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_NEUTRAL).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_NEUTRAL).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_NEUTRAL).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of neutral class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)
    min_expected = expected.min()

    # Fisher exact
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_neutral": low_correct,
        "low_wrong_neutral": low_wrong,
        "high_correct_neutral": high_correct,
        "high_wrong_neutral": high_wrong,

        "low_total_neutral": low_total,
        "high_total_neutral": high_total,

        "low_neutral_recall": low_recall,
        "high_neutral_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct neutral prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_neutral_recall", "high_neutral_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

summary_cols = [
    "model",
    "low_correct_neutral", "low_wrong_neutral",
    "high_correct_neutral", "high_wrong_neutral",
    "low_neutral_recall", "high_neutral_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

print(f"\nSaved to: {OUTPUT_CSV}")

Υπολογισμός Overall % positive,
Low-pressure % positive,
Difference

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_positive = (df[col] == 2).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_positive = (low_df[col] == 2).sum()
    low_total = len(low_df)

    overall_pct = total_positive / total
    low_pct = low_positive / low_total

    print(model)
    print("Overall % positive:", round(overall_pct, 4))
    print("Low-pressure % positive:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

Υπολογισμός Overall % negative,
Low-pressure % negative,
Difference

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_negative = (df[col] == 0).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_negative = (low_df[col] == 0).sum()
    low_total = len(low_df)

    overall_pct = total_negative / total
    low_pct = low_negative / low_total

    print(model)
    print("Overall % negative:", round(overall_pct, 4))
    print("Low-pressure % negative:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

Υπολογισμός Overall % neutral,
Low-pressure % neutral,
Difference

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_neutral = (df[col] == 1).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_neutral = (low_df[col] == 1).sum()
    low_total = len(low_df)

    overall_pct = total_neutral / total
    low_pct = low_neutral / low_total

    print(model)
    print("Overall % neutral:", round(overall_pct, 4))
    print("Low-pressure % neutral:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

πώς μετακινείται ΟΛΟ το sentiment distribution

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total = len(df)
    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_total = len(low_df)

    overall_pos = (df[col] == 2).sum() / total
    overall_neu = (df[col] == 1).sum() / total
    overall_neg = (df[col] == 0).sum() / total

    low_pos = (low_df[col] == 2).sum() / low_total
    low_neu = (low_df[col] == 1).sum() / low_total
    low_neg = (low_df[col] == 0).sum() / low_total

    print(model)
    print("Overall:",
          round(overall_pos,4),
          round(overall_neu,4),
          round(overall_neg,4))

    print("Low-pressure:",
          round(low_pos,4),
          round(low_neu,4),
          round(low_neg,4))

    print("Difference:",
          round(low_pos-overall_pos,4),
          round(low_neu-overall_neu,4),
          round(low_neg-overall_neg,4))

    print("-----")

ΕΚΤΙΜΗΣΗ RECALL POSITIVE CLASS ΜΟΝΟ ΓΙΑ REAL CITATIONS. ΕΞΑΙΡΟΥΝΤΑΙ ΟΙ IEEE CITATIONS.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level_only_real_citations.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/positive_low_vs_high_hpi_stats_only_real_citations.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_POSITIVE = 2

# =========================
# KEEP ONLY TRUE POSITIVE-CLASS INSTANCES
# i.e. rows where the true label is positive
# =========================
pos_df = df[df["ground_truth"] == GROUND_TRUTH_POSITIVE].copy()

# Pressure groups
low_df = pos_df[pos_df["HPI_global"] <= THRESHOLD].copy()
high_df = pos_df[pos_df["HPI_global"] > THRESHOLD].copy()

print(f"Total positive instances: {len(pos_df)}")
print(f"Low-pressure positive instances: {len(low_df)}")
print(f"High-pressure positive instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct positive = model predicted 2 when ground truth is 2
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_POSITIVE).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_POSITIVE).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_POSITIVE).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_POSITIVE).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of positive class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)

    # Decide whether small expected counts exist
    min_expected = expected.min()

    # Fisher exact test for 2x2
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    # Relative recall drop/gain from low to high
    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_positive": low_correct,
        "low_wrong_positive": low_wrong,
        "high_correct_positive": high_correct,
        "high_wrong_positive": high_wrong,

        "low_total_positive": low_total,
        "high_total_positive": high_total,

        "low_positive_recall": low_recall,
        "high_positive_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct positive prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_positive_recall", "high_positive_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

print(f"\nSaved to: {OUTPUT_CSV}")

summary_cols = [
    "model",
    "low_correct_positive", "low_wrong_positive",
    "high_correct_positive", "high_wrong_positive",
    "low_positive_recall", "high_positive_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_positive = (df[col] == 2).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_positive = (low_df[col] == 2).sum()
    low_total = len(low_df)

    overall_pct = total_positive / total
    low_pct = low_positive / low_total

    print(model)
    print("Overall % positive:", round(overall_pct, 4))
    print("Low-pressure % positive:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

ΕΚΤΙΜΗΣΗ RECALL NEGATIVE CLASS ΜΟΝΟ ΓΙΑ REAL CITATIONS. ΕΞΑΙΡΟΥΝΤΑΙ ΟΙ IEEE CITATIONS.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level_only_real_citations.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/negative_low_vs_high_hpi_stats_only_real_citations.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_NEGATIVE = 0

# =========================
# KEEP ONLY TRUE NEGATIVE-CLASS INSTANCES
# i.e. rows where the true label is negative
# =========================
neg_df = df[df["ground_truth"] == GROUND_TRUTH_NEGATIVE].copy()

# Pressure groups
low_df = neg_df[neg_df["HPI_global"] <= THRESHOLD].copy()
high_df = neg_df[neg_df["HPI_global"] > THRESHOLD].copy()

print(f"Total negative instances: {len(neg_df)}")
print(f"Low-pressure negative instances: {len(low_df)}")
print(f"High-pressure negative instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct negative = model predicted 0 when ground truth is 0
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_NEGATIVE).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_NEGATIVE).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_NEGATIVE).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_NEGATIVE).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of negative class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)
    min_expected = expected.min()

    # Fisher exact
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_negative": low_correct,
        "low_wrong_negative": low_wrong,
        "high_correct_negative": high_correct,
        "high_wrong_negative": high_wrong,

        "low_total_negative": low_total,
        "high_total_negative": high_total,

        "low_negative_recall": low_recall,
        "high_negative_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct negative prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_negative_recall", "high_negative_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

summary_cols = [
    "model",
    "low_correct_negative", "low_wrong_negative",
    "high_correct_negative", "high_wrong_negative",
    "low_negative_recall", "high_negative_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

print(f"\nSaved to: {OUTPUT_CSV}")

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_negative = (df[col] == 0).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_negative = (low_df[col] == 0).sum()
    low_total = len(low_df)

    overall_pct = total_negative / total
    low_pct = low_negative / low_total

    print(model)
    print("Overall % negative:", round(overall_pct, 4))
    print("Low-pressure % negative:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

ΕΚΤΙΜΗΣΗ RECALL NEUTRAL CLASS ΜΟΝΟ ΓΙΑ REAL CITATIONS. ΕΞΑΙΡΟΥΝΤΑΙ ΟΙ IEEE CITATIONS.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
INPUT_CSV = "data/processed/dataset_with_HPI_global_sentence_level_only_real_citations.csv"
THRESHOLD = 2.1036
OUTPUT_CSV = "outputs/results/neutral_low_vs_high_hpi_stats_only_real_citations.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_CSV)

# =========================
# SETTINGS
# =========================
MODELS = {
    "BART": "BART_pred",
    "T5": "T5_pred",
    "SciBERT": "SciBERT_pred",
    "RoBERTa": "RoBERTa_pred",
    "DeepSeek": "DeepSeek_pred",
    "LLaMA3": "LLaMA3_pred",
}

GROUND_TRUTH_NEUTRAL = 1

# =========================
# KEEP ONLY TRUE NEUTRAL-CLASS INSTANCES
# i.e. rows where the true label is neutral
# =========================
neu_df = df[df["ground_truth"] == GROUND_TRUTH_NEUTRAL].copy()

# Pressure groups
low_df = neu_df[neu_df["HPI_global"] <= THRESHOLD].copy()
high_df = neu_df[neu_df["HPI_global"] > THRESHOLD].copy()

print(f"Total neutral instances: {len(neu_df)}")
print(f"Low-pressure neutral instances: {len(low_df)}")
print(f"High-pressure neutral instances: {len(high_df)}")

results = []

for model_name, pred_col in MODELS.items():

    # Correct neutral = model predicted 1 when ground truth is 1
    low_correct = int((low_df[pred_col] == GROUND_TRUTH_NEUTRAL).sum())
    low_wrong   = int((low_df[pred_col] != GROUND_TRUTH_NEUTRAL).sum())

    high_correct = int((high_df[pred_col] == GROUND_TRUTH_NEUTRAL).sum())
    high_wrong   = int((high_df[pred_col] != GROUND_TRUTH_NEUTRAL).sum())

    # 2x2 table:
    #                Correct   Wrong
    # Low pressure      a        b
    # High pressure     c        d
    table = np.array([
        [low_correct, low_wrong],
        [high_correct, high_wrong]
    ])

    # Recall of neutral class in each region
    low_total = low_correct + low_wrong
    high_total = high_correct + high_wrong

    low_recall = low_correct / low_total if low_total > 0 else np.nan
    high_recall = high_correct / high_total if high_total > 0 else np.nan

    # Chi-square
    chi2, chi2_p, dof, expected = chi2_contingency(table, correction=False)
    min_expected = expected.min()

    # Fisher exact
    fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

    # Safer odds ratio with Haldane-Anscombe correction if any zero cell
    a, b = low_correct, low_wrong
    c, d = high_correct, high_wrong

    if 0 in [a, b, c, d]:
        a_adj, b_adj, c_adj, d_adj = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_adj, b_adj, c_adj, d_adj = a, b, c, d

    odds_ratio_correct_low_vs_high = (a_adj * d_adj) / (b_adj * c_adj)

    recall_diff = high_recall - low_recall

    results.append({
        "model": model_name,

        "low_correct_neutral": low_correct,
        "low_wrong_neutral": low_wrong,
        "high_correct_neutral": high_correct,
        "high_wrong_neutral": high_wrong,

        "low_total_neutral": low_total,
        "high_total_neutral": high_total,

        "low_neutral_recall": low_recall,
        "high_neutral_recall": high_recall,
        "recall_difference_high_minus_low": recall_diff,

        "chi2": chi2,
        "chi2_p": chi2_p,
        "dof": dof,
        "min_expected_count": min_expected,

        "fisher_odds_ratio": fisher_or,
        "fisher_p": fisher_p,

        # OR > 1 means better odds of correct neutral prediction in LOW vs HIGH
        "odds_ratio_correct_low_vs_high": odds_ratio_correct_low_vs_high,

        "recommended_test": "Fisher" if min_expected < 5 else "Chi-square"
    })

# =========================
# MULTIPLE TESTING CORRECTION
# =========================
results_df = pd.DataFrame(results)

# BH correction for chi-square p-values
chi2_reject, chi2_p_bh, _, _ = multipletests(results_df["chi2_p"], method="fdr_bh")
results_df["chi2_p_bh"] = chi2_p_bh
results_df["chi2_significant_bh"] = chi2_reject

# BH correction for Fisher p-values
fisher_reject, fisher_p_bh, _, _ = multipletests(results_df["fisher_p"], method="fdr_bh")
results_df["fisher_p_bh"] = fisher_p_bh
results_df["fisher_significant_bh"] = fisher_reject

# =========================
# ROUND FOR READABILITY
# =========================
float_cols = [
    "low_neutral_recall", "high_neutral_recall", "recall_difference_high_minus_low",
    "chi2", "chi2_p", "min_expected_count",
    "fisher_odds_ratio", "fisher_p",
    "odds_ratio_correct_low_vs_high",
    "chi2_p_bh", "fisher_p_bh"
]

for col in float_cols:
    results_df[col] = results_df[col].astype(float).round(6)

# =========================
# SAVE + PRINT
# =========================
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n=== RESULTS ===")
print(results_df)

summary_cols = [
    "model",
    "low_correct_neutral", "low_wrong_neutral",
    "high_correct_neutral", "high_wrong_neutral",
    "low_neutral_recall", "high_neutral_recall",
    "chi2_p", "chi2_p_bh",
    "fisher_p", "fisher_p_bh",
    "odds_ratio_correct_low_vs_high",
    "recommended_test"
]

print("\n=== PAPER-READY SUMMARY ===")
print(results_df[summary_cols])

print(f"\nSaved to: {OUTPUT_CSV}")

In [ ]:
THRESHOLD = 2.1036

for model, col in MODELS.items():

    total_neutral = (df[col] == 1).sum()
    total = len(df)

    low_df = df[df["HPI_global"] <= THRESHOLD]
    low_neutral = (low_df[col] == 1).sum()
    low_total = len(low_df)

    overall_pct = total_neutral / total
    low_pct = low_neutral / low_total

    print(model)
    print("Overall % neutral:", round(overall_pct, 4))
    print("Low-pressure % neutral:", round(low_pct, 4))
    print("Difference:", round(low_pct - overall_pct, 4))
    print("-----")

Διάβασμα του Step3_Final_Dataset_with_HI_Scores.csv προκειμένου να πάρω τους συντελεστές για να φτιάξω τα πινακάκια. Το αρχείο αυτό περιλαμβάνει μόνο citations που έχουν μια μόνο αναφορά-πηγή.

In [ ]:
df = pd.read_csv('data/processed/Step3_Final_Dataset_with_HI_Scores.csv')

μέσοι όροι HPI Scores

In [ ]:
average_bart_hi_score = df['BART_HI_Score'].mean()
print(f"Ο μέσος όρος του BART_HI_Score είναι: {average_bart_hi_score}")

In [ ]:
average_roberta_hi_score = df['RoBERTa_HI_Score'].mean()
print(f"Ο μέσος όρος του RoBERTa_HI_Score είναι: {average_roberta_hi_score}")

In [ ]:
average_t5_hi_score = df['T5_HI_Score'].mean()
print(f"Ο μέσος όρος του T5_HI_Score είναι: {average_t5_hi_score}")

αξιζει να μελετηθoυν και οι περιπτωσεις : 1. οπου ο HI ειναι αρνητικος , και αρα υπαρχει πιεση ωστε να γινει σωστη προβλεψη απο το μοντελο αλλα τελικα αυτο κανει λαθος προβλεψη. 2. οπου ο HI ειναι αρνητικος , και αρα υπαρχει πιεση ωστε να γινει σωστη προβλεψη απο το μοντελο και τελικα αυτο κανει σωστη προβλεψη. 3. οπου ο HI ειναι θετικος , και αρα υπαρχει πιεση ωστε να γινει λαθος προβλεψη απο το μοντελο και τελικα κανει λαθος προβλεψη. 4. οπου ο HI ειναι θετικος , και αρα υπαρχει πιεση ωστε να γινει λαθος προβλεψη απο το μοντελο αλλα τελικα αυτο κανει σωστη προβλεψη.

In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

results = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    #print(df[hi_col].dtype)

    if hi_col in df.columns and err_col in df.columns:
        # Case 1: HI <= 0 and Error = 1 (Unexpected Error)
        case1 = df[(df[hi_col] <= 0) & (df[err_col] == 1)].shape[0]
        # Case 2: HI <= 0 and Error = 0 (Expected Success)
        case2 = df[(df[hi_col] <= 0) & (df[err_col] == 0)].shape[0]
        # Case 3: HI > 0 and Error = 1 (Expected Error)
        case3 = df[(df[hi_col] > 0) & (df[err_col] == 1)].shape[0]
        # Case 4: HI > 0 and Error = 0 (Unexpected Success)
        case4 = df[(df[hi_col] > 0) & (df[err_col] == 0)].shape[0]

        total = df.shape[0]

        results.append({
            'Model': model,
            'Case 1 (HI<=0, Err=1)': case1,
            'Case 2 (HI<=0, Err=0)': case2,
            'Case 3 (HI>0, Err=1)': case3,
            'Case 4 (HI>0, Err=0)': case4
        })

results_df = pd.DataFrame(results)
results_df.head(6)

stacked bar chart

In [ ]:
import matplotlib.pyplot as plt

# Build the stacked chart directly from the case counts computed above.
case_plot_df = results_df.copy()
case_plot_df = case_plot_df.rename(columns={
    'Case 1 (HI<=0, Err=1)': 'Case 1',
    'Case 2 (HI<=0, Err=0)': 'Case 2',
    'Case 3 (HI>0, Err=1)': 'Case 3',
    'Case 4 (HI>0, Err=0)': 'Case 4',
}).set_index('Model')

case_plot_pct = case_plot_df.div(case_plot_df.sum(axis=1), axis=0) * 100

ax = case_plot_pct.plot(kind='barh', stacked=True, figsize=(10, 6))
plt.title('Distribution of Cases per Model (%)', fontsize=14)
plt.xlabel('Percentage (%)')
plt.ylabel('Model')
plt.legend(title='Cases', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('outputs/figures/fig_comparable/stacked_bars.pdf')
plt.show()


In [ ]:
LLaMA3_error_zero_hi = df[(df['LLaMA3_is_error'] == 1) & (df['LLaMA3_HI_Score'] <= 0)].shape[0]
print(f"Σύνολο εγγραφών για LLaMA3_is_error = 1 και LLaMA3_HI_Score = 0: {LLaMA3_error_zero_hi}")

 Nα απομονώσουμε τις 20 περιπτώσεις του T5 (Case 1) και την 1 περιπτωση του llama3 για να δούμε τι πήγε στραβά ενώ όλα έδειχναν ότι θα ήταν σωστό.

In [ ]:
# Filter Case 1 for T5
t5_case1 = df[(df['T5_HI_Score'] < 0) & (df['T5_is_error'] == 1)]

# Filter Case 1 for LLaMA3
llama3_case1 = df[(df['LLaMA3_HI_Score'] < 0) & (df['LLaMA3_is_error'] == 1)]

# Display T5 results
print("--- T5 Case 1 (HI < 0, Error = 1) ---")
cols_to_show = ['sentence', 'ground_truth', 'T5_pred', 'T5_HI_Score', 'Semantic_Depth', 'Negation_Count', 'T5_shap_pos_abs_dist']
print(t5_case1[cols_to_show])

# Display LLaMA3 results
print("\n--- LLaMA3 Case 1 (HI < 0, Error = 1) ---")
llama_cols = ['sentence', 'ground_truth', 'LLaMA3_pred', 'LLaMA3_HI_Score', 'Semantic_Depth', 'Negation_Count', 'LLAMA3_shap_pos_abs_dist']
print(llama3_case1[llama_cols])

# Save to CSV for the user to download if needed
combined_case1 = pd.concat([t5_case1, llama3_case1])
combined_case1.to_csv('outputs/results/Case1_Unexpected_Errors.csv', index=False)

In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

case_comparison = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"
    gap_col = f"{model}_shap_pos_abs_dist" if f"{model}_shap_pos_abs_dist" in df.columns else f"{model.upper()}_shap_pos_abs_dist"

    # Case 3: HI > 0, Error = 1 (Pressure leads to Error)
    c3_mask = (df[hi_col] > 0) & (df[err_col] == 1)
    # Case 4: HI > 0, Error = 0 (Pressure overcome - Robust)
    c4_mask = (df[hi_col] > 0) & (df[err_col] == 0)

    if c3_mask.any():
        c3_stats = {
            'Model': model,
            'Case': 'Case 3 (Lapse)',
            'Count': c3_mask.sum(),
            'Mean_HI': df.loc[c3_mask, hi_col].mean(),
            'Mean_Depth': df.loc[c3_mask, 'Semantic_Depth'].mean(),
            'Mean_Gap': df.loc[c3_mask, gap_col].mean(),
            'Mean_Negation': df.loc[c3_mask, 'Negation_Count'].mean()
        }
        case_comparison.append(c3_stats)

    if c4_mask.any():
        c4_stats = {
            'Model': model,
            'Case': 'Case 4 (Robust)',
            'Count': c4_mask.sum(),
            'Mean_HI': df.loc[c4_mask, hi_col].mean(),
            'Mean_Depth': df.loc[c4_mask, 'Semantic_Depth'].mean(),
            'Mean_Gap': df.loc[c4_mask, gap_col].mean(),
            'Mean_Negation': df.loc[c4_mask, 'Negation_Count'].mean()
        }
        case_comparison.append(c4_stats)

comparison_df = pd.DataFrame(case_comparison)
comparison_df.head(12)

In [ ]:
# Επιλέγουμε μόνο T5 και LLaMA3
selected_models = ['T5', 'LLaMA3']

case_comparison = []

for model in selected_models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"
    gap_col = (
        f"{model}_shap_pos_abs_dist"
        if f"{model}_shap_pos_abs_dist" in df.columns
        else f"{model.upper()}_shap_pos_abs_dist"
    )

    # -------------------------
    # Case 1: HI <= 0, Error = 1 (Unexpected Error)
    # -------------------------
    c1_mask = (df[hi_col] <= 0) & (df[err_col] == 1)

    if c1_mask.any():
        c1_stats = {
            'Model': model,
            'Case': 'Case 1 (Error)',
            'Count': c1_mask.sum(),
            'Mean_HI': df.loc[c1_mask, hi_col].mean(),
            'Mean_Depth': df.loc[c1_mask, 'Semantic_Depth'].mean(),
            'Mean_Gap': df.loc[c1_mask, gap_col].mean(),
            'Mean_Negation': df.loc[c1_mask, 'Negation_Count'].mean()
        }
        case_comparison.append(c1_stats)

    # -------------------------
    # Case 2: HI <= 0, Error = 0 (Protective / No Pressure)
    # -------------------------
    c2_mask = (df[hi_col] <= 0) & (df[err_col] == 0)

    if c2_mask.any():
        c2_stats = {
            'Model': model,
            'Case': 'Case 2 (Protective)',
            'Count': c2_mask.sum(),
            'Mean_HI': df.loc[c2_mask, hi_col].mean(),
            'Mean_Depth': df.loc[c2_mask, 'Semantic_Depth'].mean(),
            'Mean_Gap': df.loc[c2_mask, gap_col].mean(),
            'Mean_Negation': df.loc[c2_mask, 'Negation_Count'].mean()
        }
        case_comparison.append(c2_stats)

comparison_df = pd.DataFrame(case_comparison)
comparison_df.head(12)

Κανόνας εμπιστοσύνης. Για κάθε μοντέλο:
παίρνουμε μόνο Case 4 (HI>0 & Error=0)
υπολογίζουμε:

P75(HI) → ανεκτή πίεση

P90(HI) → υψηλή πίεση

P95(HI) → υπερβολική πίεση (εκτός “robust range”)

In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

trust_rules = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Case 4: HI > 0 & Error = 0 (Robust under pressure)
    c4_mask = (df[hi_col] > 0) & (df[err_col] == 0)

    if not c4_mask.any():
        continue

    hi_vals = df.loc[c4_mask, hi_col]

    stats = {
        'Model': model,
        'Case 4': 'Robust',
        'Sentence_Count': c4_mask.sum(),
        'HI_P75': hi_vals.quantile(0.75),
        'HI_P90': hi_vals.quantile(0.90),
        'HI_P95': hi_vals.quantile(0.95),
        'HI_Max': hi_vals.max()
    }

    trust_rules.append(stats)

trust_df = pd.DataFrame(trust_rules)
trust_df


για μια οποιαδήποτε πρόταση, στον κώδικα πιο κάτω βάζω το μοντέλο
που θέλω και το HI της πρότασης που θέλω να ελέγξω.

In [ ]:
row = trust_df[trust_df['Model'] == 'T5'].iloc[0]

# example
HI = 3.45

if HI <= row['HI_P75']:
    decision = "Safe (within robust pressure range)"  # Ασφαλές (εντός εύρους ανθεκτικής πίεσης)
elif HI <= row['HI_P95']:
    decision = "Caution (high pressure)"              # Προσοχή (υψηλή πίεση)
else:
    decision = "High risk (beyond robust capacity)"   # Υψηλός κίνδυνος (πέραν της ισχυρής χωρητικότητας)

print(decision)

Για Case 2
πόσο “ισχυρή” είναι η προστατευτική ζώνη όταν το μοντέλο προβλέπει σωστά. Αν οι τιμές του HI στο Case 2 είναι π.χ. από -3.5 έως -0.1, τότε:

τιμές κοντά στο 0 (π.χ. -0.05) είναι οριακή προστασία

πολύ αρνητικές τιμές (π.χ. -2.5) είναι ισχυρή προστασία

Προσοχή: τα P75/P90/P95 εδώ είναι “ανάποδα” διαισθητικά

Επειδή οι τιμές είναι αρνητικές:

P95 θα είναι συνήθως πολύ κοντά στο 0 (λιγότερο αρνητικό), γιατί το 95% είναι “κάτω” από αυτό (πιο αρνητικό).

P05 θα είναι το “βαρύ” αρνητικό άκρο.

Άρα, για Case 2 συχνά είναι πιο χρήσιμα:

P05, P10, P25 (δείχνουν “ισχυρή προστασία”)
και επίσης

P90, P95 (δείχνουν “οριακή προστασία”, κοντά στο 0)


Πώς το διαβάζεις

HI_Max στο Case2 θα είναι η τιμή πιο κοντά στο 0 (π.χ. -0.001).

HI_P95 επίσης θα είναι κοντά στο 0.

HI_P05 θα είναι η “βαριά” αρνητική προστασία.

In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

case2_rules = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Case 2: HI <= 0 & correct (err=0)
    c2_mask = (df[hi_col] <= 0) & (df[err_col] == 0)

    if not c2_mask.any():
        continue

    hi_vals = df.loc[c2_mask, hi_col]

    stats = {
        'Model': model,
        'Case 2': 'Protection',
        'Sentence_Count': c2_mask.sum(),
        # strong protection (left tail)
        'HI_P05': hi_vals.quantile(0.05),
        'HI_P10': hi_vals.quantile(0.10),
        'HI_P25': hi_vals.quantile(0.25),
        # near-boundary protection (close to 0)
        'HI_P75': hi_vals.quantile(0.75),
        'HI_P90': hi_vals.quantile(0.90),
        'HI_P95': hi_vals.quantile(0.95),
        'HI_Max': hi_vals.max()  # θα είναι το "closest to 0"
    }

    case2_rules.append(stats)

case2_df = pd.DataFrame(case2_rules)
case2_df

λογική για HI < 0 (Case 2: protective & correct). εδώ όσο πιο αρνητικός ο HI ⇒ τόσο μεγαλύτερη προστασία

άρα τα thresholds “διαβάζονται ανάποδα” σε σχέση με Case 4.

Κανόνας εμπιστοσύνης για Case 2 (HI < 0, προστατευτικό)
Λογική

Χρησιμοποιούμε percentiles από το Case 2 distribution:

HI_P05 → πολύ ισχυρή προστασία

HI_P25 → ισχυρή προστασία

HI_P90 / HI_P95 → οριακή προστασία (κοντά στο 0)

In [ ]:
# Παίρνουμε τα thresholds του Case 2 για το T5 ή llama3 μόνο
row = case2_df[case2_df['Model'] == 'T5'].iloc[0]

# example
HI = -1.98

if HI <= row['HI_P05']:
    decision = "Very safe (strong protective alignment)"      # Πολύ ασφαλές (ισχυρή προστασία)
elif HI <= row['HI_P25']:
    decision = "Safe (protective regime)"                     # Ασφαλές (προστατευτικό καθεστώς)
elif HI <= row['HI_P90']:
    decision = "Moderate protection (near boundary)"          # Μέτρια προστασία (κοντά στο όριο)
else:
    decision = "Weak protection (boundary case)"              # Αδύναμη προστασία (οριακή περίπτωση)

print(decision)


In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

trust_rules = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Case 3: HI > 0 & Error = 1 (error)
    c3_mask = (df[hi_col] > 0) & (df[err_col] == 1)

    if not c3_mask.any():
        continue

    hi_vals = df.loc[c3_mask, hi_col]

    stats = {
        'Model': model,
        'Case 3': 'Error',
        'Sentence_Count': c3_mask.sum(),
        'HI_P75': hi_vals.quantile(0.75),
        'HI_P90': hi_vals.quantile(0.90),
        'HI_P95': hi_vals.quantile(0.95),
        'HI_Max': hi_vals.max()
    }

    trust_rules.append(stats)

trust_df = pd.DataFrame(trust_rules)
trust_df


In [ ]:
hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

case1_rules = []

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Case 1: HI <= 0 & correct (err=1)
    c1_mask = (df[hi_col] <= 0) & (df[err_col] == 1)

    if not c1_mask.any():
        continue

    hi_vals = df.loc[c1_mask, hi_col]

    stats = {
        'Model': model,
        'Case 1': 'Error',
        'Sentence_Count': c1_mask.sum(),
        # strong protection (left tail)
        'HI_P05': hi_vals.quantile(0.05),
        'HI_P10': hi_vals.quantile(0.10),
        'HI_P25': hi_vals.quantile(0.25),
        # near-boundary protection (close to 0)
        'HI_P75': hi_vals.quantile(0.75),
        'HI_P90': hi_vals.quantile(0.90),
        'HI_P95': hi_vals.quantile(0.95),
        'HI_Max': hi_vals.max()  # θα είναι το "closest to 0"
    }

    case1_rules.append(stats)

case1_df = pd.DataFrame(case1_rules)
case1_df

boxplot που δείχνει καθαρά τη μετατόπιση της κατανομής. Σταθερός άξονας y.
CASE 4 -> CASE 3.
3 μοντέλα ανα γραμμή.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Plot: Case 4 vs Case 3 — 2x3 GRID (single figure)
# --------------------------------------------------

def plot_hi_shift_boxplots_grid(df, models, out_dir=None, show_points=True, dpi=200):
    """
    2x3 grid:
      - Each subplot = one model
      - Inside each subplot: Case 4 (Correct) vs Case 3 (Error)
      - Shared GLOBAL y-axis limits for paper consistency
    """

    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------
    # FIXED GLOBAL Y-AXIS (paper-consistent)
    # --------------------------------------------------
    global_y_min = 0.0
    global_y_max = 6.0
    step = 0.5
    global_yticks = np.arange(global_y_min, global_y_max + 1e-9, step)

    # --------------------------------------------------
    # Create figure (2 rows x 3 columns)
    # --------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(9, 10), sharey=True)
    axes = axes.flatten()

    rng = np.random.default_rng(7)

    for ax, m in zip(axes, models):

        hi_col = f"{m}_HI_Score"
        err_col = f"{m}_is_error"

        if hi_col not in df.columns or err_col not in df.columns:
            ax.set_visible(False)
            continue

        c4 = df[(df[hi_col] > 0) & (df[err_col] == 0)][hi_col].dropna().values
        c3 = df[(df[hi_col] > 0) & (df[err_col] == 1)][hi_col].dropna().values

        if len(c4) < 5 or len(c3) < 5:
            ax.set_visible(False)
            continue

        data = [c4, c3]
        labels = [f"Case 4\n(n={len(c4)})", f"Case 3\n(n={len(c3)})"]

        ax.boxplot(
            data,
            tick_labels=labels,
            showfliers=True,
            widths=0.5
        )

        # Optional jittered points
        if show_points:
            for i, arr in enumerate(data, start=1):
                x = rng.normal(loc=i, scale=0.04, size=len(arr))
                ax.scatter(x, arr, s=10, alpha=0.25)

        # Percentiles
        c4_p75, c4_p95 = np.percentile(c4, [75, 95])
        c3_p75, c3_p95 = np.percentile(c3, [75, 95])

        ax.hlines([c4_p75, c4_p95], xmin=0.85, xmax=1.15,
                  linestyles="--", linewidth=1)
        ax.hlines([c3_p75, c3_p95], xmin=1.85, xmax=2.15,
                  linestyles="--", linewidth=1)

        ax.text(1.18, c4_p75, "P75", va="center", fontsize=10)
        ax.text(1.18, c4_p95, "P95", va="center", fontsize=10)
        ax.text(2.18, c3_p75, "P75", va="center", fontsize=10)
        ax.text(2.18, c3_p95, "P95", va="center", fontsize=10)

        ax.set_ylim(global_y_min, global_y_max)
        ax.set_yticks(global_yticks)
        ax.set_title(m, fontsize=12)
        ax.grid(True, axis="y", alpha=0.25)

    # Shared labels
    fig.supylabel("HPI Score", fontsize=10)
    fig.suptitle("HPI Distribution Shift (Case 4 → Case 3)", fontsize=15)

    plt.tight_layout(rect=[0, 0.02, 1, 0.95])

    if out_dir is not None:
        fname = os.path.join(out_dir, "HPI_Shift_Boxplot_2x3_grid.pdf")
        plt.savefig(fname, format="pdf", bbox_inches="tight")
        print(f"[SAVED] {fname}")

    plt.show()


models = ['BART', 'T5', 'SciBERT', 'RoBERTa', 'DeepSeek', 'LLaMA3']
out_dir = 'outputs/figures/HPI_Shift_fixed_y'

plot_hi_shift_boxplots_grid(
    df,
    models,
    out_dir=out_dir,
    show_points=True
)

boxplot που δείχνει καθαρά τη μετατόπιση της κατανομής. Σταθερός άξονας y.
CASE 4 -> CASE 3

In [ ]:
# ---------------------------------------------
# Plot: Case 4 (HI>0 & correct) vs Case 3 (HI>0 & error)
# One boxplot figure per model, with GLOBAL y-axis limits
# ---------------------------------------------

def plot_hi_shift_boxplots(df, models, out_dir=None, show_points=True, dpi=200):
    """
    For each model:
      - Case 4: HI > 0 & is_error == 0  (Robust under pressure)
      - Case 3: HI > 0 & is_error == 1  (Pressure leads to error)
    Produces a single boxplot figure per model showing distribution shift.

    Uses GLOBAL y-axis limits (computed once from all models' HI>0 values)
    so all figures share the same y-axis for direct comparison.

    Parameters
    ----------
    df : pd.DataFrame
    models : list[str]
    out_dir : str | None
        If provided, saves each figure as PDF into out_dir.
    show_points : bool
        If True, overlay jittered points (helps show sample sizes / overlap).
    dpi : int
        Ignored for PDF (vector), kept for compatibility.
    """
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------
    # GLOBAL HI limits (shared y-axis for all models)
    # We use ALL HI>0 values across models (regardless of correct/error)
    # --------------------------------------------------
    all_hi_vals = []
    for m in models:
        hi_col = f"{m}_HI_Score"
        err_col = f"{m}_is_error"  # just to ensure the model exists consistently

        if hi_col in df.columns and err_col in df.columns:
            vals = df[df[hi_col] > 0][hi_col].dropna().values
            if len(vals) > 0:
                all_hi_vals.append(vals)

    if len(all_hi_vals) == 0:
        raise ValueError("Δεν βρέθηκαν HI τιμές > 0 για κανένα μοντέλο. Έλεγξε τις στήλες *_HI_Score.")

    """all_hi_vals = np.concatenate(all_hi_vals)
    global_y_min = np.percentile(all_hi_vals, 1)
    global_y_max = np.percentile(all_hi_vals, 99)


    # Force y-axis to start at 0 (since we plot HI > 0)
    global_y_min = 0.0

    # Make the top limit "nice" (e.g., 4.13 -> 4.2 or 4.5)
    step = 0.5  # change to 0.2 if you want finer ticks
    global_y_max = np.ceil(global_y_max / step) * step

    # Paper-friendly ticks
    global_yticks = np.round(np.arange(global_y_min, global_y_max + 1e-9, step), 2)"""


    # --------------------------------------------------
    # FIXED GLOBAL Y-AXIS (paper-consistent)
    # --------------------------------------------------
    global_y_min = 0.0
    global_y_max = 6.0

    step = 0.5
    global_yticks = np.arange(global_y_min, global_y_max + 1e-9, step)


    # --------------------------------------------------
    # Per-model plots
    # --------------------------------------------------
    for m in models:
        hi_col = f"{m}_HI_Score"
        err_col = f"{m}_is_error"

        if hi_col not in df.columns or err_col not in df.columns:
            print(f"[SKIP] {m}: missing {hi_col} or {err_col}")
            continue

        # Define cases based on your definitions
        c4 = df[(df[hi_col] > 0) & (df[err_col] == 0)][hi_col].dropna().values  # Case 4
        c3 = df[(df[hi_col] > 0) & (df[err_col] == 1)][hi_col].dropna().values  # Case 3

        if len(c4) < 5 or len(c3) < 5:
            print(f"[SKIP] {m}: too few samples (Case4={len(c4)}, Case3={len(c3)})")
            continue

        # ---- Plot
        plt.figure(figsize=(7, 5))
        data = [c4, c3]
        labels = [f"Case 4 (Correct)\n(n={len(c4)})", f"Case 3 (Error)\n(n={len(c3)})"]

        # Matplotlib 3.9+ uses tick_labels instead of labels
        plt.boxplot(
            data,
            tick_labels=labels,
            showfliers=True,   # keep outliers visible
            widths=0.5
        )

        # Optional: overlay jittered points (shows overlap + sample density)
        if show_points:
            rng = np.random.default_rng(7)
            for i, arr in enumerate(data, start=1):
                x = rng.normal(loc=i, scale=0.045, size=len(arr))  # jitter
                plt.scatter(x, arr, s=10, alpha=0.25)

        # Add percentile markers (paper-friendly “shift” cues)
        c4_p75, c4_p95 = np.percentile(c4, [75, 95])
        c3_p75, c3_p95 = np.percentile(c3, [75, 95])

        # Horizontal lines across each box position (short segments)
        plt.hlines([c4_p75, c4_p95], xmin=0.85, xmax=1.15, linestyles="--", linewidth=1)
        plt.hlines([c3_p75, c3_p95], xmin=1.85, xmax=2.15, linestyles="--", linewidth=1)

        # Labels for those lines
        plt.text(1.18, c4_p75, "P75", va="center", fontsize=9)
        plt.text(1.18, c4_p95, "P95", va="center", fontsize=9)
        plt.text(2.18, c3_p75, "P75", va="center", fontsize=9)
        plt.text(2.18, c3_p95, "P95", va="center", fontsize=9)

        # GLOBAL shared y-axis limits + ticks
        plt.ylim(global_y_min, global_y_max)
        plt.yticks(global_yticks)

        plt.ylabel("HPI Score")
        plt.title(f"HPI Distribution Shift (Case 4 → Case 3) — {m}")
        plt.grid(True, axis="y", alpha=0.25)
        plt.tight_layout()

        if out_dir is not None:
            fname = os.path.join(out_dir, f"HPI_Shift_Boxplot_case4_case3_{m}.pdf")
            plt.savefig(fname, format="pdf", bbox_inches="tight")
            print(f"[SAVED] {fname}")

        plt.show()



models = ['BART', 'T5', 'SciBERT', 'RoBERTa', 'DeepSeek', 'LLaMA3']
out_dir = 'outputs/figures/HPI_Shift_fixed_y'
plot_hi_shift_boxplots(df, models, out_dir=out_dir, show_points=True)

boxplot που δείχνει καθαρά τη μετατόπιση της κατανομής. Μη σταθερός άξονας y.
CASE 4 -> CASE 3

In [ ]:
# ---------------------------------------------
# Plot: Case 4 (HI>0 & correct) vs Case 3 (HI>0 & error)
# One boxplot figure per model
# ---------------------------------------------

def plot_hi_shift_boxplots(df, models, out_dir=None, show_points=True, dpi=200):
    """
    For each model:
      - Case 4: HI > 0 & is_error == 0  (Robust under pressure)
      - Case 3: HI > 0 & is_error == 1  (Pressure leads to error)
    Produces a single boxplot figure per model showing distribution shift.

    Parameters
    ----------
    df : pd.DataFrame
    models : list[str]
    out_dir : str | None
        If provided, saves each figure as PNG into out_dir.
    show_points : bool
        If True, overlay jittered points (helps show sample sizes / overlap).
    dpi : int
    """
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    for m in models:
        hi_col = f"{m}_HI_Score"
        err_col = f"{m}_is_error"

        if hi_col not in df.columns or err_col not in df.columns:
            print(f"[SKIP] {m}: missing {hi_col} or {err_col}")
            continue

        # Define cases based on your own definitions
        c4 = df[(df[hi_col] > 0) & (df[err_col] == 0)][hi_col].dropna().values  # Case 4
        c3 = df[(df[hi_col] > 0) & (df[err_col] == 1)][hi_col].dropna().values  # Case 3

        if len(c4) < 5 or len(c3) < 5:
            print(f"[SKIP] {m}: too few samples (Case4={len(c4)}, Case3={len(c3)})")
            continue

        # Common y-limits for fair visual comparison (robust to outliers via percentiles)
        all_vals = np.concatenate([c4, c3])
        y_min = np.percentile(all_vals, 1)
        y_max = np.percentile(all_vals, 99)
        pad = 0.05 * (y_max - y_min) if (y_max > y_min) else 0.1
        y_min, y_max = y_min - pad, y_max + pad

        # ---- Plot
        plt.figure(figsize=(9, 6))
        data = [c4, c3]
        labels = [f"Case 4 (Correct)\n(n={len(c4)})", f"Case 3 (Error)\n(n={len(c3)})"]

        bp = plt.boxplot(
            data,
            tick_labels=labels,
            showfliers=True,   # keep outliers visible
            widths=0.5
        )

        # Optional: overlay jittered points (shows overlap + sample density)
        if show_points:
            rng = np.random.default_rng(7)
            for i, arr in enumerate(data, start=1):
                x = rng.normal(loc=i, scale=0.045, size=len(arr))  # jitter
                plt.scatter(x, arr, s=10, alpha=0.25)

        # Add median/percentile markers (paper-friendly “shift” cues)
        c4_p75, c4_p95 = np.percentile(c4, [75, 95])
        c3_p75, c3_p95 = np.percentile(c3, [75, 95])

        # Horizontal lines across each box position (short segments)
        plt.hlines([c4_p75, c4_p95], xmin=0.85, xmax=1.15, linestyles="--", linewidth=1)
        plt.hlines([c3_p75, c3_p95], xmin=1.85, xmax=2.15, linestyles="--", linewidth=1)

        # Labels for those lines
        plt.text(1.18, c4_p75, "P75", va="center", fontsize=9)
        plt.text(1.18, c4_p95, "P95", va="center", fontsize=9)
        plt.text(2.18, c3_p75, "P75", va="center", fontsize=9)
        plt.text(2.18, c3_p95, "P95", va="center", fontsize=9)

        plt.ylim(y_min, y_max)
        plt.ylabel("HPI Score")
        plt.title(f"HPI Distribution Shift (Case 4 → Case 3) — {m}")
        plt.grid(True, axis="y", alpha=0.25)
        plt.tight_layout()

        if out_dir is not None:
            fname = os.path.join(out_dir, f"HPI_Shift_Boxplot_case4_case3_{m}.pdf")
            plt.savefig(fname, format="pdf", bbox_inches="tight")
            print(f"[SAVED] {fname}")

        plt.show()


models = ['BART', 'T5', 'SciBERT', 'RoBERTa', 'DeepSeek', 'LLaMA3']
out_dir = 'outputs/figures/HPI_Shift_no_fixed_y'  # μετατόπιση της κατανομής.
plot_hi_shift_boxplots(df, models, out_dir=out_dir, show_points=True)

να κάνουμε έναν γρήγορο έλεγχο στις λέξεις-κλειδιά (SHAP Keywords) της Case 4 για να δούμε αν "βλέπουν" λογική ή αν "κοιτάνε" απλά ονόματα. Θα γινει για ολα τα μοντελα. Το κάνω επειδή υπάρχουν στην case 4 σωστές προβλέψεις, ενώ το μοντέλο έχει υποστεί πιέση να κάνει λάθος πρόβλεψη. Οπότε ίσως υπάρξουν σωστές προβλέψεις αλλά για λάθος λόγους (τυχαίες).

In [ ]:
from collections import Counter

hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

# Dictionary to store top keywords for Case 4 per model
case4_keywords = {}

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"
    # Column naming inconsistency check
    kw_col = f"{model}_shap_keyword_token"
    if kw_col not in df.columns:
         kw_col = f"{model.upper()}_shap_keyword_token"

    if hi_col in df.columns and err_col in df.columns and kw_col in df.columns:
        # Filter Case 4: HI > 0 and Correct (is_error == 0)
        c4_mask = (df[hi_col] > 0) & (df[err_col] == 0)
        keywords = df.loc[c4_mask, kw_col].dropna().astype(str).tolist()

        # Clean keywords a bit (strip whitespace)
        keywords = [k.strip() for k in keywords]

        counts = Counter(keywords).most_common(15)
        case4_keywords[model] = counts

# Print results in a readable way
for model, top_kws in case4_keywords.items():
    print(f"--- {model} Case 4 Top SHAP Keywords ---")
    for kw, count in top_kws:
        print(f"  {kw}: {count}")
    print("\n")

Case 3 (HI>0, Err=1)  Υπάρχει πίεση και το μοντέλο κάνει λάθος. Αναμενόμενη συμπεριφορά. Απλά θέλω να διαπιστώσω άν για το λάθος φταίει μόνο η δομική πίεση του HI, ή μπορεί να φταίνε και άσχετα tokens SHAP σε κάθε πρόταση.

In [ ]:
from collections import Counter

hi_cols = [col for col in df.columns if 'HI_Score' in col]
models = [col.replace('_HI_Score', '') for col in hi_cols]

# Dictionary to store top keywords for Case 3 per model
case3_keywords = {}

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Column naming inconsistency check
    kw_col = f"{model}_shap_keyword_token"
    if kw_col not in df.columns:
        kw_col = f"{model.upper()}_shap_keyword_token"

    if hi_col in df.columns and err_col in df.columns and kw_col in df.columns:
        # Filter Case 3: HI > 0 and Error (is_error == 1)
        c3_mask = (df[hi_col] > 0) & (df[err_col] == 1)

        keywords = df.loc[c3_mask, kw_col].dropna().astype(str).tolist()

        # Clean keywords a bit
        keywords = [k.strip() for k in keywords]

        counts = Counter(keywords).most_common(15)
        case3_keywords[model] = counts

# Print results
for model, top_kws in case3_keywords.items():
    print(f"--- {model} Case 3 Top SHAP Keywords ---")
    for kw, count in top_kws:
        print(f"  {kw}: {count}")
    print("\n")


Case 2 (HI <= 0 & correct). Εδώ υπάρχει προστασία απο το HI και το μοντελο οντως προβλεπει σωστά. Το ερώτημα είναι προβλέπει σωστά μόνον λόγω της προστασίας ή συμβάλλει στην επιτυχία του μοντέλου και το περιεχόμενο της πρότασης?

In [ ]:
from collections import Counter

# Ρητός ορισμός μοντέλων για Case 2
models = ["T5", "LLaMA3"]

case2_keywords = {}

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Column naming inconsistency check
    kw_col = f"{model}_shap_keyword_token"
    if kw_col not in df.columns:
        kw_col = f"{model.upper()}_shap_keyword_token"

    if hi_col in df.columns and err_col in df.columns and kw_col in df.columns:
        # Filter Case 2: HI <= 0 and Correct (is_error == 0)
        c2_mask = (df[hi_col] <= 0) & (df[err_col] == 0)

        keywords = df.loc[c2_mask, kw_col].dropna().astype(str).tolist()
        keywords = [k.strip() for k in keywords]

        counts = Counter(keywords).most_common(15)
        case2_keywords[model] = counts

# Print results
for model, top_kws in case2_keywords.items():
    print(f"--- {model} Case 2 Top SHAP Keywords ---")
    for kw, count in top_kws:
        print(f"  {kw}: {count}")
    print("\n")


Case 1 = HI <= 0 & λάθος πρόβλεψη. Εδώ υπάρχει προστασία απο τον δομικό δείκτη ΗΙ, ωστόσο το μοντέλο κάνει λάθος πρόβλεψη. Το ερώτημα είναι, εφόσον υπάρχει προστασία, μήπως το λάθος οφείλεται στο περιεχόμενος της πρότασης?

In [ ]:
from collections import Counter

# Ρητός ορισμός μοντέλων για Case 1
models = ["T5", "LLaMA3"]

case1_keywords = {}

for model in models:
    hi_col = f"{model}_HI_Score"
    err_col = f"{model}_is_error"

    # Column naming inconsistency check
    kw_col = f"{model}_shap_keyword_token"
    if kw_col not in df.columns:
        kw_col = f"{model.upper()}_shap_keyword_token"

    if hi_col in df.columns and err_col in df.columns and kw_col in df.columns:
        # Filter Case 1: HI <= 0 and Error (is_error == 1)
        c1_mask = (df[hi_col] <= 0) & (df[err_col] == 1)

        keywords = df.loc[c1_mask, kw_col].dropna().astype(str).tolist()
        keywords = [k.strip() for k in keywords]

        counts = Counter(keywords).most_common(15)
        case1_keywords[model] = counts

# Print results
for model, top_kws in case1_keywords.items():
    print(f"--- {model} Case 1 Top SHAP Keywords ---")
    for kw, count in top_kws:
        print(f"  {kw}: {count}")
    print("\n")


END